# 🏐 BetEdge PoC — LNV Women (FR_LAF_W)
## Baseline Engine v0 + Typical Lineups v0 + Lineup Shock v0

**Uruchom komórki po kolei.** Każda sekcja jest oznaczona fazą (A–I).

| Faza | Opis |
|---|---|
| A | Normalizacja zawodniczek → `players_master` |
| B | Player Impact Score per mecz |
| C | Rolling baseline → `player_pis_rolling` |
| D | Typowe składy → `expected_lineups` |
| E | Baseline drużyny → `team_baselines` |
| F | Struktura live lineups + mock loader |
| G | Lineup Shock Engine |
| H | Korekta fair odds |
| I | Alert log → `alerts_log` |

## ⚙️ CELL 1 — Instalacja bibliotek

In [ ]:
# ── Instalacja ────────────────────────────────────────────────
!pip install gspread google-auth pandas numpy rapidfuzz python-dateutil pydantic --quiet
print('✅ Biblioteki zainstalowane')

## 🔧 KONFIGURACJA — Stałe i parametry

In [ ]:
import json, os, time, logging, warnings
import pandas as pd
import numpy as np
from datetime import datetime, date
from typing import Dict, List, Optional, Tuple, Any
import gspread
from google.oauth2.service_account import Credentials
warnings.filterwarnings("ignore")

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s %(levelname)s %(message)s",
    datefmt="%H:%M:%S"
)
log = logging.getLogger("betedge")

# ── Wczytaj service_account.json z panelu bocznego ────────────
with open("/content/service_account.json", "r") as f:
    SA_INFO = json.load(f)
print(f"✅ Wczytano credentials: {SA_INFO.get('client_email', '?')}")

# ── Stałe ─────────────────────────────────────────────────────
SPREADSHEET_ID = "1x80zxUeYfKk9_kvsgeR1D9ayoKfaTPE90wNOGvuodnI"
LEAGUE_KEY     = "FR_LAF_W"
SCOPES = [
    "https://www.googleapis.com/auth/spreadsheets",
    "https://www.googleapis.com/auth/drive",
]

# ── Nazwy zakładek ─────────────────────────────────────────────
SH_MATCHES         = "matches"
SH_PLAYERS_RAW     = "players_raw"
SH_PLAYERS_MASTER  = "players_master"
SH_TEAM_BASELINES  = "team_baselines"
SH_PIS_ROLLING     = "player_pis_rolling"
SH_TYPICAL_LINEUPS = "expected_lineups"
SH_LIVE_LINEUPS    = "live_lineups"
SH_ALERTS_LOG      = "alerts_log"

# ── Throttling ─────────────────────────────────────────────────
DELAY_WRITE = 1.2
BATCH_SIZE  = 40

# ── PIS Coefficients ───────────────────────────────────────────
PIS_COEFF = {
    "att_pts": 1.00,
    "blocks":  1.30,
    "srv_ace": 1.20,
    "srv_err": -0.80,
    "att_err": -0.80,
}
LIBERO_REC_THRESHOLD  = 0.50
LIBERO_REC_MULTIPLIER = 4.0

# ── Rolling weights ────────────────────────────────────────────
PIS_W_SEASON = 0.5
PIS_W_LAST5  = 0.3
PIS_W_LAST3  = 0.2

# ── Role weights ───────────────────────────────────────────────
ROLE_WEIGHTS = {
    "S":       1.25,
    "OPP":     1.20,
    "OH":      1.10,
    "MB":      1.00,
    "L":       0.90,
    "UNKNOWN": 0.85,
}
KEY_ROLES = {"S", "OPP", "OH", "L"}

# ── Lineup slots ───────────────────────────────────────────────
LINEUP_SLOTS = [
    ("S",   "S",   1),
    ("OPP", "OPP", 1),
    ("OH1", "OH",  1),
    ("OH2", "OH",  2),
    ("MB1", "MB",  1),
    ("MB2", "MB",  2),
    ("L",   "L",   1),
]

# ── Lineup Shock thresholds ────────────────────────────────────
SHOCK_THRESHOLDS = {
    "neutral_or_positive": -0.05,
    "low":    -0.08,
    "medium": -0.15,
}

# ── Fair odds bounds ───────────────────────────────────────────
MAX_SHOCK_ADJUSTMENT = 0.12
SHOCK_SENSITIVITY    = 0.60

# ── Edge alert bands ───────────────────────────────────────────
EDGE_WATCH        = 0.02
EDGE_VALUE        = 0.04
EDGE_STRONG_ALERT = 0.06

print("✅ Konfiguracja załadowana — wszystko gotowe!")


## 🛠️ HELPERS — Funkcje pomocnicze
Uruchom raz. Dostępne we wszystkich fazach.

In [ ]:
import unicodedata, hashlib, re, math, uuid, os, time, logging, warnings
import pandas as pd
import numpy as np
from datetime import datetime, date
from typing import Dict, List, Optional, Tuple, Any
import gspread
from google.oauth2.service_account import Credentials
warnings.filterwarnings("ignore")

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(message)s", datefmt="%H:%M:%S")
log = logging.getLogger("betedge")

# ═══════════════════════════════════════════════════════════════
# HELPERS — wszystkie funkcje pomocnicze w jednym miejscu
# Uruchamiane raz, dostępne we wszystkich komórkach poniżej.
# ═══════════════════════════════════════════════════════════════

# ── Normalizacja nazw ─────────────────────────────────────────
def normalize_name(raw: str) -> str:
    if not raw or not isinstance(raw, str):
        return ""
    name = raw.strip()
    name = re.sub(r"\s+", " ", name)
    nfd  = unicodedata.normalize("NFD", name)
    name = "".join(c for c in nfd if unicodedata.category(c) != "Mn")
    return name.title().strip()

def make_team_id(team_name: str) -> str:
    key = normalize_name(team_name)
    return "T_" + hashlib.md5(key.encode()).hexdigest()[:8].upper()

def make_player_id(normalized_name: str, team_id: str) -> str:
    key = f"{normalized_name}|{team_id}"
    return "P_" + hashlib.md5(key.encode()).hexdigest()[:10].upper()

def safe_float(val, default: float = 0.0) -> float:
    try:
        f = float(str(val).strip().replace(",", "."))
        return f if np.isfinite(f) else default
    except (ValueError, TypeError):
        return default

def confidence_from_matches(n: int) -> str:
    if n <= 2:  return "low"
    if n <= 4:  return "medium"
    return "normal"

def sigmoid(x: float) -> float:
    try:
        return 1.0 / (1.0 + math.exp(-x))
    except OverflowError:
        return 0.0 if x < 0 else 1.0

def clamp(val: float, lo: float = 0.01, hi: float = 0.99) -> float:
    return max(lo, min(hi, val))

# ── Google Sheets helpers ─────────────────────────────────────
def df_to_rows(df: pd.DataFrame) -> list:
    return df.fillna("").astype(str).values.tolist()

def ensure_worksheet(book, name: str, rows: int = 500, cols: int = 30) -> gspread.Worksheet:
    try:
        return book.worksheet(name)
    except gspread.exceptions.WorksheetNotFound:
        log.info(f"  Tworzę zakładkę: {name}")
        return book.add_worksheet(title=name, rows=rows, cols=cols)

def batch_append(ws: gspread.Worksheet, rows: list, delay: float = 1.2) -> None:
    for i in range(0, len(rows), BATCH_SIZE):
        chunk = rows[i:i + BATCH_SIZE]
        for attempt in range(3):
            try:
                ws.append_rows(chunk, value_input_option="USER_ENTERED")
                break
            except gspread.exceptions.APIError as e:
                if "429" in str(e) and attempt < 2:
                    time.sleep(60)
                else:
                    raise
        if i + BATCH_SIZE < len(rows):
            time.sleep(delay)

def clear_and_write(ws: gspread.Worksheet, headers: list, rows: list) -> None:
    ws.clear()
    time.sleep(1.2)
    ws.append_row(headers, value_input_option="USER_ENTERED")
    time.sleep(1.2)
    if rows:
        batch_append(ws, rows)
    log.info(f"  → {ws.title}: {len(rows)} wierszy")

def save_df(book, sheet_name: str, df: pd.DataFrame) -> bool:
    if df is None or df.empty:
        print(f"  ⚠️  POMINIĘTO {sheet_name} — pusty DataFrame")
        return False
    try:
        ws = ensure_worksheet(book, sheet_name, rows=max(500, len(df)+10), cols=len(df.columns)+2)
        ws.clear()
        time.sleep(1.2)
        ws.update("A1", [list(df.columns)], value_input_option="USER_ENTERED")
        time.sleep(1.2)
        rows = df_to_rows(df)
        for i in range(0, len(rows), 40):
            ws.append_rows(rows[i:i+40], value_input_option="USER_ENTERED")
            if i + 40 < len(rows):
                time.sleep(1.2)
        print(f"  ✅ {sheet_name:<30} → {len(df)} wierszy, {len(df.columns)} kol.")
        return True
    except Exception as e:
        print(f"  ❌ BŁĄD {sheet_name}: {e}")
        return False

# ── Fair odds helpers ─────────────────────────────────────────
def adjust_probability(base_prob: float, delta_tis_pct: float, shock_level: str) -> float:
    raw_adj    = delta_tis_pct * SHOCK_SENSITIVITY
    bounded    = max(-MAX_SHOCK_ADJUSTMENT, min(MAX_SHOCK_ADJUSTMENT, raw_adj))
    model_prob = base_prob + bounded
    return round(max(0.05, min(0.95, model_prob)), 4)

def compute_edge_band(edge: float, shock_level: str) -> str:
    if edge > EDGE_STRONG_ALERT and shock_level in ("medium", "high"):
        return "strong_alert"
    elif edge > EDGE_VALUE:
        return "value"
    elif edge > EDGE_WATCH:
        return "watch"
    return "no_bet"

def compute_fair_odds(shock_result: dict, base_prob: float, market_odds=None) -> dict:
    delta_pct   = float(shock_result.get("delta_TIS_pct", 0.0))
    shock_level = shock_result.get("shock_level", "neutral_or_positive")
    model_prob  = adjust_probability(base_prob, delta_pct, shock_level)
    true_odds   = round(1.0 / model_prob, 3) if model_prob > 0 else 999.0
    market_prob = edge = None
    edge_band   = "no_data"
    if market_odds and float(market_odds) > 1.0:
        market_prob = round(1.0 / float(market_odds), 4)
        edge        = round(model_prob - market_prob, 4)
        edge_band   = compute_edge_band(edge, shock_level)
    return {
        "base_prob": base_prob, "model_prob": model_prob,
        "true_odds": true_odds, "market_prob": market_prob,
        "market_odds": market_odds, "edge": edge, "edge_band": edge_band,
    }

def decision_label(edge: float) -> str:
    if edge is None:      return "NO ODDS"
    if edge >= EDGE_STRONG_ALERT: return "STRONG BET"
    if edge >= EDGE_VALUE:        return "BET"
    if edge >= EDGE_WATCH:        return "WATCH"
    return "NO BET"

print("✅ HELPERS załadowane")


## 🔑 Połączenie z Google Sheets

In [ ]:
# ── Połączenie z Sheets ────────────────────────────────────────
def connect_sheets() -> gspread.Spreadsheet:
    creds  = Credentials.from_service_account_info(SA_INFO, scopes=SCOPES)
    client = gspread.authorize(creds)
    return client.open_by_key(SPREADSHEET_ID)

def safe_write(ws: gspread.Worksheet, range_a1: str, values: list, retries: int = 3) -> None:
    """Zapis z retry przy 429 quota."""
    for attempt in range(retries):
        try:
            ws.update(range_a1, values, value_input_option='USER_ENTERED')
            return
        except gspread.exceptions.APIError as e:
            if '429' in str(e) and attempt < retries - 1:
                wait = 60 * (attempt + 1)
                log.warning(f'429 quota — czekam {wait}s...')
                time.sleep(wait)
            else:
                raise

def batch_append(ws: gspread.Worksheet, rows: list) -> None:
    """Batch append z throttlingiem."""
    for i in range(0, len(rows), BATCH_SIZE):
        chunk = rows[i:i + BATCH_SIZE]
        for attempt in range(3):
            try:
                ws.append_rows(chunk, value_input_option='USER_ENTERED')
                break
            except gspread.exceptions.APIError as e:
                if '429' in str(e) and attempt < 2:
                    time.sleep(60)
                else:
                    raise
        if i + BATCH_SIZE < len(rows):
            time.sleep(DELAY_WRITE)

def clear_and_write(ws: gspread.Worksheet, headers: list, rows: list) -> None:
    """Idempotentny zapis: czyści arkusz, wstawia nagłówki + dane."""
    ws.clear()
    time.sleep(DELAY_WRITE)
    ws.append_row(headers, value_input_option='USER_ENTERED')
    time.sleep(DELAY_WRITE)
    if rows:
        batch_append(ws, rows)
    log.info(f'  → {ws.title}: {len(rows)} wierszy zapisanych')

def ws_to_df(ws: gspread.Worksheet) -> pd.DataFrame:
    """Wczytaj worksheet jako DataFrame (nagłówki z row 1)."""
    data = ws.get_all_values()
    if len(data) < 2:
        return pd.DataFrame()
    return pd.DataFrame(data[1:], columns=data[0])

BOOK = connect_sheets()
print(f'✅ Połączono z arkuszem: {BOOK.title}')

## 📥 CELL 3 — Wczytanie danych źródłowych

In [ ]:
def load_raw_data(book: gspread.Spreadsheet) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """Wczytuje matches i players_raw z Google Sheets."""
    log.info('📥 Wczytuję matches...')
    matches_ws = book.worksheet(SH_MATCHES)
    matches_df = ws_to_df(matches_ws)
    log.info(f'   matches: {len(matches_df)} wierszy, kolumny: {list(matches_df.columns)}')

    time.sleep(DELAY_WRITE)

    log.info('📥 Wczytuję players_raw...')
    players_ws = book.worksheet(SH_PLAYERS_RAW)
    players_df = ws_to_df(players_ws)
    log.info(f'   players_raw: {len(players_df)} wierszy, kolumny: {list(players_df.columns)}')

    return matches_df, players_df

matches_df, players_raw_df = load_raw_data(BOOK)

print(f'\n📊 Podsumowanie danych:')
print(f'  Mecze:      {len(matches_df)}')
print(f'  Zawodniczki (wiersze): {len(players_raw_df)}')
if len(players_raw_df) > 0:
    print(f'  Kolumny players_raw:  {list(players_raw_df.columns)}')
    print(players_raw_df.head(3))

## 🏷️ FAZA A0 — Teams Master → `teams_master`

Słownik drużyn: jedna drużyna = jeden `team_id`.
Rozwiązuje chaos nazewniczy (Beziers / Béziers / Beziers Women → jedna nazwa kanoniczna).

In [ ]:
TEAM_NAME_OVERRIDES = {
    # Le Cannet
    "Le Cannet W":                  "Volero Le Cannet",
    "Le Cannet":                    "Volero Le Cannet",
    "Volero Le Cannet":             "Volero Le Cannet",
    # Vandoeuvre Nancy
    "Vandoeuvre Nancy W":           "Vandoeuvre Nancy VB",
    "Vandoeuvre Nancy":             "Vandoeuvre Nancy VB",
    "Vandoeuvre-Nancy":             "Vandoeuvre Nancy VB",
    # Mulhouse
    "Mulhouse W":                   "Volley Mulhouse Alsace",
    "Mulhouse":                     "Volley Mulhouse Alsace",
    "ASPTT Mulhouse":               "Volley Mulhouse Alsace",
    # Terville Florange
    "Terville Florange W":          "Terville-Florange OC",
    "Terville Florange":            "Terville-Florange OC",
    "Terville-Florange OC":         "Terville-Florange OC",
    "Terville-Florange":            "Terville-Florange OC",
    # France Avenir
    "France Avenir W":              "France Avenir",
    "France Avenir":                "France Avenir",
    "France Avenir 2024":           "France Avenir",
    # Beziers
    "Béziers W":                    "Beziers Volley",
    "Béziers":                      "Beziers Volley",
    "Beziers W":                    "Beziers Volley",
    "Beziers":                      "Beziers Volley",
    "Beziers Volley":               "Beziers Volley",
    # Nantes
    "Nantes W":                     "Neptunes de Nantes",
    "Nantes":                       "Neptunes de Nantes",
    "Neptunes Nantes":              "Neptunes de Nantes",
    "Neptunes de Nantes":           "Neptunes de Nantes",
    # Levallois Paris
    "Levallois Paris W":            "Levallois Paris SC",
    "Levallois Paris":              "Levallois Paris SC",
    "Levallois Paris SC":           "Levallois Paris SC",
    "Levallois Paris Saint-Cloud":  "Levallois Paris SC",
    # RC Cannes
    "RC Cannes W":                  "RC Cannes",
    "RC Cannes":                    "RC Cannes",
    "Cannes W":                     "RC Cannes",
    "Cannes":                       "RC Cannes",
    # Venelles
    "Venelles W":                   "Pays d Aix Venelles",
    "Venelles":                     "Pays d Aix Venelles",
    "Pays d Aix Venelles":          "Pays d Aix Venelles",
    "Pays d'Aix Venelles":         "Pays d Aix Venelles",
    # Marcq en Baroeul
    "Marcq en Baroeul W":           "VC Marcq-en-Baroeul",
    "Marcq en Baroeul":             "VC Marcq-en-Baroeul",
    "Marcq-en-Baroeul W":           "VC Marcq-en-Baroeul",
    "Marcq-en-Baroeul":             "VC Marcq-en-Baroeul",
    "VC Marcq-en-Baroeul":          "VC Marcq-en-Baroeul",
    "Marcq En Baroeul":             "VC Marcq-en-Baroeul",
    "Marcq En Baroeul W":           "VC Marcq-en-Baroeul",
    # Evreux
    "Evreux W":                     "Evreux Volley-Ball",
    "Evreux":                       "Evreux Volley-Ball",
    "Evreux Volley-Ball":           "Evreux Volley-Ball",
    # Saint-Die
    "St-Dié W":                     "Saint-Die-des-Vosges VB",
    "St-Dié":                       "Saint-Die-des-Vosges VB",
    "Saint-Die W":                  "Saint-Die-des-Vosges VB",
    "Saint-Die":                    "Saint-Die-des-Vosges VB",
    "St-Die W":                     "Saint-Die-des-Vosges VB",
    "St-Die":                       "Saint-Die-des-Vosges VB",
    # Bordeaux
    "Bordeaux Mérignac W":          "Bordeaux Merignac Volley",
    "Bordeaux Mérignac":            "Bordeaux Merignac Volley",
    "Bordeaux Merignac W":          "Bordeaux Merignac Volley",
    "Bordeaux Merignac":            "Bordeaux Merignac Volley",
    "Bordeaux Merignac Volley":     "Bordeaux Merignac Volley",
    # Chamalieres
    "Chamalières W":                "VBC Chamalieres",
    "Chamalières":                  "VBC Chamalieres",
    "Chamalieres W":                "VBC Chamalieres",
    "Chamalieres":                  "VBC Chamalieres",
    "VBC Chamalieres":              "VBC Chamalieres",
}

# ── Cache dla teams_master ────────────────────────────────────
# Załadowany raz, używany przez get_canonical_team_name() wszędzie
_TEAMS_MASTER_CACHE: Dict[str, str] = {}  # team_id → canonical_name
_TEAMS_RAW_CACHE:    Dict[str, str] = {}  # normalized_raw_name → canonical_name

def load_teams_master_cache(book) -> None:
    """Wczytaj teams_master ze Sheets i zbuduj cache canonical names."""
    global _TEAMS_MASTER_CACHE, _TEAMS_RAW_CACHE
    try:
        ws   = book.worksheet("teams_master")
        data = ws.get_all_values()
        if len(data) < 2:
            return
        hdrs = data[0]
        # Kolumna C = team_name_canonical (index 2), A = team_id, D = team_name_source variants
        tid_col = hdrs.index("team_id")          if "team_id"          in hdrs else 0
        can_col = hdrs.index("team_name_canonical") if "team_name_canonical" in hdrs else 2
        raw_col = hdrs.index("team_name_source")    if "team_name_source"    in hdrs else None

        for row in data[1:]:
            if len(row) <= max(tid_col, can_col):
                continue
            tid      = str(row[tid_col]).strip()
            canonical= str(row[can_col]).strip()
            if not canonical or canonical in ("team_name_canonical", ""):
                continue
            if tid:
                _TEAMS_MASTER_CACHE[tid] = canonical
            # Also map raw variants → canonical
            _TEAMS_RAW_CACHE[normalize_name(canonical)] = canonical
            if raw_col and len(row) > raw_col:
                for variant in str(row[raw_col]).split("|"):
                    v = variant.strip()
                    if v:
                        _TEAMS_RAW_CACHE[normalize_name(v)] = canonical

        log.info(f"teams_master cache: {len(_TEAMS_MASTER_CACHE)} team_id → canonical, "
                 f"{len(_TEAMS_RAW_CACHE)} raw variants")
    except Exception as e:
        log.warning(f"Nie udało się załadować teams_master cache: {e}")

def get_canonical_team_name(raw_name: str) -> str:
    """Zwraca kanoniczną nazwę drużyny z teams_master (kolumna C).
    Fallback: TEAM_NAME_OVERRIDES → normalize_name."""
    if not raw_name or not isinstance(raw_name, str):
        return "UNKNOWN"
    stripped = raw_name.strip()

    # 1. Szukaj w raw cache z teams_master
    norm = normalize_name(stripped)
    if norm in _TEAMS_RAW_CACHE:
        return _TEAMS_RAW_CACHE[norm]

    # 2. Szukaj w TEAM_NAME_OVERRIDES (legacy fallback)
    if stripped in TEAM_NAME_OVERRIDES:
        return TEAM_NAME_OVERRIDES[stripped]
    for key, val in TEAM_NAME_OVERRIDES.items():
        if normalize_name(key) == norm:
            return val

    # 3. Może to hash — sprawdź _TID_FALLBACK
    if re.match(r"^T_[0-9A-F]{8}$", stripped):
        return _TID_FALLBACK.get(stripped, stripped)
    # 4. Fallback
    return normalize_name(stripped)

# Pełna hardcoded mapa wszystkich znanych hashów → canonical name
_TID_FALLBACK: Dict[str, str] = {
    "T_000A8F2D":"RC Cannes W","T_10F6443E":"Volero Le Cannet W",
    "T_17DB9AF2":"Vandoeuvre Nancy W","T_1B61891D":"Saint-Die W",
    "T_1BCCE2E1":"France Avenir W","T_1DF6D6E8":"Terville-Florange W",
    "T_26F8D1BA":"RC Cannes W","T_2ABCCC91":"VC Marcq-en-Baroeul W",
    "T_3573710B":"Volero Le Cannet W","T_3A39AE4D":"VBC Chamalieres W",
    "T_3E95ADAF":"Volley Mulhouse Alsace W","T_42D6AC8D":"VBC Chamalieres W",
    "T_4478BD8E":"Saint-Die W","T_448452C8":"Saint-Die W",
    "T_46ED19C8":"Vandoeuvre Nancy W","T_4D37EA0A":"VC Marcq-en-Baroeul W",
    "T_52B9E374":"VC Marcq-en-Baroeul W","T_563494AD":"Evreux W",
    "T_5715F6F5":"Bordeaux Merignac W","T_5E826293":"Terville-Florange W",
    "T_680FC345":"Vandoeuvre Nancy W","T_7702338A":"Beziers W",
    "T_760A407A":"Levallois Paris W","T_7A8B6A2E":"Terville-Florange W",
    "T_7109C8DB":"Terville-Florange W",  # already there"T_9349D15E":"Levallois Paris W",
    "T_97C5F67F":"VBC Chamalieres W","T_984A5CA5":"VC Marcq-en-Baroeul W",
    "T_A6DE60EE":"Bordeaux Merignac W","T_AC85909C":"RC Cannes W",
    "T_AF0458E5":"Vandoeuvre Nancy W","T_B3D6B895":"RC Cannes W",
    "T_B4D7CFFE":"Levallois Paris W","T_B5461916":"France Avenir W",
    "T_BAC26741":"VC Marcq-en-Baroeul W","T_C08D4FD7":"Evreux W",
    "T_C239C558":"Beziers W","T_C2E1BAF6":"Volley Mulhouse Alsace W",
    "T_CAE3526E":"Saint-Die W","T_D1A550BA":"Volero Le Cannet W",
    "T_DF7569AD":"Saint-Die W","T_E2C3736B":"Volley Mulhouse Alsace W",
    "T_ED8EC9A4":"Terville-Florange W","T_F0D580EC":"Bordeaux Merignac W",
    "T_FA261CEF":"France Avenir W","T_FE511BE1":"Beziers W",
}

def get_canonical_name_by_tid(team_id: str) -> str:
    """Zwraca kanoniczną nazwę drużyny po team_id.
    Priorytet: teams_master cache → hardcoded fallback → TEAM_NAME_OVERRIDES → team_id
    """
    tid = str(team_id).strip()
    # 1. Cache z teams_master (najaktualniejszy)
    if tid in _TEAMS_MASTER_CACHE:
        return _TEAMS_MASTER_CACHE[tid]
    # 2. Hardcoded fallback (wszystkie znane hashe)
    if tid in _TID_FALLBACK:
        return _TID_FALLBACK[tid]
    # 3. Jeśli wygląda jak hash T_XXXXXXXX — zgłoś i zwróć "UNKNOWN"
    if re.match(r"^T_[0-9A-F]{8}$", tid):
        log.warning(f"Nieznany team_id: {tid} — dodaj do _TID_FALLBACK lub teams_master")
        return "UNKNOWN"
    # 4. Może to już jest nazwa — przepuść przez get_canonical_team_name
    return get_canonical_team_name(tid)

def sync_teams_master(players_raw: pd.DataFrame, book) -> pd.DataFrame:
    """Faza A0: Synchronizuje teams_master w Sheets.

    ZASADA: team_name_canonical (kolumna C) jest TYLKO do ręcznej edycji przez użytkownika.
    Notebook NIGDY jej nie nadpisuje.

    Co robi:
    - Wczytuje aktualną teams_master ze Sheets
    - Dodaje nowe drużyny (których jeszcze nie ma) z team_name_source = raw names
    - Aktualizuje team_name_source dla istniejących drużyn (dodaje nowe warianty)
    - NIE dotyka team_name_canonical
    """
    cols = list(players_raw.columns)
    c_team = next((c for c in cols if "team_name" in c.lower()), None)
    if not c_team:
        log.warning("Brak kolumny team_name w players_raw")
        return pd.DataFrame()

    # Zbierz wszystkie surowe nazwy z players_raw
    all_raw = players_raw[c_team].fillna("").astype(str).str.strip().unique().tolist()
    all_raw = [r for r in all_raw if r]

    # Grupuj warianty po team_id (hash z raw name przez TEAM_NAME_OVERRIDES)
    tid_to_variants: dict = {}
    tid_to_canonical_guess: dict = {}
    for raw in all_raw:
        canonical_guess = get_canonical_team_name(raw)
        t_id = make_team_id(canonical_guess)
        if t_id not in tid_to_variants:
            tid_to_variants[t_id] = set()
            tid_to_canonical_guess[t_id] = canonical_guess
        tid_to_variants[t_id].add(raw)

    # Wczytaj aktualną teams_master ze Sheets
    try:
        ws   = book.worksheet("teams_master")
        data = ws.get_all_values()
        hdrs = data[0] if data else []
        existing_rows = data[1:] if len(data) > 1 else []
    except Exception:
        ws   = book.add_worksheet("teams_master", rows=50, cols=10)
        hdrs = []
        existing_rows = []

    # Sprawdź kolumny
    needed_cols = ["team_id","league_code","team_name_canonical",
                   "team_name_source","country","active_flag"]
    if not hdrs:
        hdrs = needed_cols
        ws.update("A1", [hdrs], value_input_option="USER_ENTERED")
        existing_rows = []

    # Indeksy kolumn
    def ci(col): return hdrs.index(col) if col in hdrs else None
    i_tid  = ci("team_id")              or 0
    i_can  = ci("team_name_canonical")  or 2   # kolumna C — TYLKO CZYTAJ
    i_src  = ci("team_name_source")     or 3
    i_lc   = ci("league_code")          or 1

    # Zbuduj mapę istniejących wierszy
    existing = {}  # team_id → row_index (1-based in sheet = row+2)
    existing_data = {}
    for j, row in enumerate(existing_rows):
        if len(row) > i_tid:
            t = str(row[i_tid]).strip()
            if t:
                existing[t] = j + 2  # sheet row number
                existing_data[t] = row

    new_rows = []
    updates  = []  # (row_num, col_num, value)

    for t_id, variants in sorted(tid_to_variants.items()):
        src_str = " | ".join(sorted(variants))
        if t_id in existing:
            # Drużyna istnieje — zaktualizuj TYLKO team_name_source (nie canonical!)
            row_num = existing[t_id]
            old_row = existing_data[t_id]
            old_src = str(old_row[i_src]).strip() if len(old_row) > i_src else ""
            # Dodaj nowe warianty których jeszcze nie ma
            old_variants = set(v.strip() for v in old_src.split("|") if v.strip())
            all_variants = old_variants | variants
            new_src = " | ".join(sorted(all_variants))
            if new_src != old_src:
                updates.append((row_num, i_src + 1, new_src))  # 1-indexed col
        else:
            # Nowa drużyna — dodaj wiersz BEZ wypełniania team_name_canonical
            # (użytkownik sam wpisze właściwą nazwę)
            new_row = [""] * len(hdrs)
            new_row[i_tid] = t_id
            new_row[i_lc]  = LEAGUE_KEY
            # team_name_canonical (i_can) = PUSTY — do ręcznego wypełnienia
            new_row[i_can] = ""
            new_row[i_src] = src_str
            new_rows.append(new_row)

    # Zastosuj aktualizacje
    import time
    if updates:
        for row_num, col_num, val in updates:
            col_letter = chr(ord("A") + col_num - 1)
            ws.update(f"{col_letter}{row_num}", [[val]], value_input_option="USER_ENTERED")
            time.sleep(0.3)
        log.info(f"teams_master: zaktualizowano team_name_source dla {len(updates)} drużyn")

    if new_rows:
        ws.append_rows(new_rows, value_input_option="USER_ENTERED")
        log.info(f"teams_master: dodano {len(new_rows)} nowych drużyn — "
                 f"wypełnij team_name_canonical ręcznie!")

    # Wczytaj aktualny stan (z Twoimi canonical names)
    data2 = ws.get_all_values()
    df = pd.DataFrame(data2[1:], columns=data2[0]) if len(data2) > 1 else pd.DataFrame()
    return df

# ── Uruchom ────────────────────────────────────────────────────
print("🅰️ Faza A0: Synchronizacja teams_master...")
teams_master_df = sync_teams_master(players_raw_df, BOOK)

print(f"✅ teams_master: {len(teams_master_df)} drużyn")
if not teams_master_df.empty:
    print(teams_master_df[["team_id","team_name_canonical","team_name_source"]].to_string(index=False))

# ── Załaduj cache z TWOICH canonical names ───────────────────
load_teams_master_cache(BOOK)
print(f"✅ Cache: {len(_TEAMS_MASTER_CACHE)} team_id → canonical name")
if _TEAMS_MASTER_CACHE:
    for t, n in sorted(_TEAMS_MASTER_CACHE.items())[:5]:
        print(f"   {t} → {n}")


## 🅰️ FAZA A — Normalizacja → `players_master`

In [ ]:
from rapidfuzz import fuzz, process

# ── Helpers normalizacji ───────────────────────────────────────

def infer_position(is_libero: bool, player_name_raw: str) -> Tuple[str, str]:
    """
    Określa pozycję na podstawie libero flag.
    Zwraca (position, confidence_flag).
    Na v0 nieznane pozycje → 'UNKNOWN'.
    """
    if is_libero:
        return 'L', 'high'
    return 'UNKNOWN', 'low'

FUZZY_THRESHOLD = 88   # min score rapidfuzz (0–100)

def build_alias_map(names: List[str]) -> Dict[str, str]:
    """
    Buduje słownik aliasów: {variant_name → canonical_name}.
    Używa rapidfuzz do wykrycia nazw, które są 'prawie takie same'.
    """
    canonical: Dict[str, str] = {}
    sorted_names = sorted(set(names))
    for name in sorted_names:
        if name in canonical:
            continue
        matches = process.extract(
            name, sorted_names,
            scorer=fuzz.token_sort_ratio,
            score_cutoff=FUZZY_THRESHOLD,
            limit=10
        )
        for match_name, score, _ in matches:
            if match_name != name and match_name not in canonical:
                canonical[match_name] = name   # alias → canonical
    return canonical

# ── Główna funkcja normalizacji ────────────────────────────────

def build_players_master(players_raw: pd.DataFrame) -> pd.DataFrame:
    """
    Faza A: Buduje players_master z players_raw.
    """
    if players_raw.empty:
        log.warning('players_raw jest pusty — brak danych do normalizacji')
        return pd.DataFrame()

    cols = list(players_raw.columns)
    log.info(f'Kolumny players_raw: {cols}')

    # ── Mapowanie kolumn (elastyczne — toleruje różne nazwy) ───
    col_map = {
        'player_name': next((c for c in cols if 'player_name' in c.lower()), None),
        'team_name':   next((c for c in cols if 'team_name' in c.lower() or c.lower() == 'team'), None),
        'team_side':   next((c for c in cols if 'side' in c.lower()), None),
        'match_id':    next((c for c in cols if 'match_id' in c.lower()), None),
        'libero':      next((c for c in cols if 'libero' in c.lower()), None),
        'match_datetime': next((c for c in cols if 'datetime' in c.lower()), None),
    }
    log.info(f'Mapowanie kolumn: {col_map}')

    df = players_raw.copy()

    # Wymagane kolumny
    if not col_map['player_name'] or not col_map['team_name']:
        raise ValueError('Brak wymaganych kolumn player_name / team_name w players_raw')

    df['_raw_player_name'] = df[col_map['player_name']].fillna('').astype(str)
    df['_raw_team_name']   = df[col_map['team_name']].fillna('').astype(str)
    df['_match_id']        = df[col_map['match_id']].fillna('').astype(str)  if col_map['match_id']    else ''
    df['_is_libero_raw']   = df[col_map['libero']].fillna('').astype(str)    if col_map['libero']      else ''
    df['_match_dt']        = df[col_map['match_datetime']].fillna('').astype(str) if col_map['match_datetime'] else ''

    # Filtruj puste nazwy
    df = df[df['_raw_player_name'].str.strip() != ''].copy()

    # Normalizacja
    df['_norm_name']  = df['_raw_player_name'].apply(normalize_name)
    df['_norm_team']  = df['_raw_team_name'].apply(normalize_name)
    df['_is_libero']  = df['_is_libero_raw'].str.upper().isin(['L', 'YES', 'TRUE', '1', 'LIBERO'])
    # Zawsze używaj canonical name do generowania team_id — spójność między zakładkami
    df['_canonical_team'] = df['_raw_team_name'].apply(get_canonical_team_name)
    df['_team_id']        = df['_canonical_team'].apply(make_team_id)

    # Fuzzy alias resolution (per-team, bo ta sama nazwa może być w różnych drużynach)
    alias_map: Dict[str, str] = {}
    for team_id, grp in df.groupby('_team_id'):
        team_aliases = build_alias_map(grp['_norm_name'].tolist())
        alias_map.update(team_aliases)

    df['_canonical_name'] = df['_norm_name'].apply(lambda n: alias_map.get(n, n))
    df['_player_id']      = df.apply(
        lambda r: make_player_id(r['_canonical_name'], r['_team_id']), axis=1
    )

    # ── Agregacja per player ───────────────────────────────────
    records = []
    for (player_id, team_id), grp in df.groupby(['_player_id', '_team_id']):
        is_libero = grp['_is_libero'].any()
        position, pos_conf = infer_position(is_libero, '')

        # Daty meczów
        match_ids   = sorted(grp['_match_id'].unique().tolist())
        match_count = len(match_ids)

        conf = confidence_from_matches(match_count)
        # Jeśli pozycja UNKNOWN → niższy confidence
        if position == 'UNKNOWN':
            conf_map = {'normal': 'medium', 'medium': 'low', 'low': 'low'}
            conf = conf_map.get(conf, conf)

        records.append({
            'player_id':            player_id,
            'normalized_player_name': grp['_canonical_name'].iloc[0],
            'raw_player_name':      grp['_raw_player_name'].iloc[0],
            'current_team_name':    grp['_norm_team'].iloc[0],
            'team_id':              team_id,
            'position':             position,
            'is_libero':            str(is_libero),
            'first_seen_match':     match_ids[0]  if match_ids else '',
            'last_seen_match':      match_ids[-1] if match_ids else '',
            'matches_count':        match_count,
            'status':               'active',
            'confidence_flag':      conf,
        })

    master = pd.DataFrame(records)
    log.info(f'players_master: {len(master)} zawodniczek z {master["team_id"].nunique()} drużyn')
    return master

# ── Uruchom ────────────────────────────────────────────────────
players_master_df = build_players_master(players_raw_df)
print(f'\n✅ Faza A zakończona: {len(players_master_df)} zawodniczek')
print(players_master_df[['normalized_player_name','current_team_name','position','is_libero','matches_count','confidence_flag']].head(10))

## 🎯 FAZA A2 — Przypisanie pozycji zawodniczek

**Krok 1:** Uruchom pierwszą komórkę → heurystyka automatyczna ze statystyk 
**Krok 2:** Pobierz CSV → wypełnij ręcznie brakujące pozycje → wgraj z powrotem 
**Krok 3:** Uruchom drugą komórkę → załaduj korekty

In [ ]:
def infer_positions_from_stats(players_raw: pd.DataFrame) -> pd.DataFrame:
    """
    Heurystyka v0 — przypisuje pozycje na podstawie średnich statystyk:

    Logika:
      L   → is_libero = True
      MB  → avg_blocks >= 0.8  (blokerzy mają dużo bloków)
      OPP → avg_att_pts najwyższe w drużynie  (atakujące przyjmująca)
      S   → avg_att_pts < 1.5 AND avg_blocks < 0.6  (rozgrywające mało atakują)
      OH  → pozostałe (przyjmujące atakujące)
    """
    cols = list(players_raw.columns)

    def col(kw):
        return next((c for c in cols if kw in c.lower()), None)

    c_name   = col('player_name')
    c_team   = col('team_name')
    c_match  = col('match_id')
    c_libero = col('libero')
    c_att    = col('att_pts')
    c_blocks = col('blocks')
    c_rec_t  = col('rec_tot')
    c_rec_p  = col('rec_pos')
    c_srv_a  = col('srv_ace')

    df = players_raw.copy()
    df['_norm_name'] = df[c_name].fillna('').apply(normalize_name)
    df['_norm_team'] = df[c_team].fillna('').apply(normalize_name)
    df['_canonical_team'] = df[c_team].fillna('').apply(get_canonical_team_name)
    df['_team_id']   = df['_canonical_team'].apply(make_team_id)
    df['_is_libero'] = df[c_libero].fillna('').astype(str).str.upper().isin(['L','YES','TRUE','1','LIBERO']) if c_libero else False
    df['_att_pts']   = df[c_att].apply(safe_float)    if c_att    else 0.0
    df['_blocks']    = df[c_blocks].apply(safe_float) if c_blocks else 0.0
    df['_rec_tot']   = df[c_rec_t].apply(safe_float)  if c_rec_t  else 0.0
    df['_rec_pos']   = df[c_rec_p].apply(safe_float)  if c_rec_p  else 0.0
    df['_srv_ace']   = df[c_srv_a].apply(safe_float)  if c_srv_a  else 0.0

    # Agreguj średnie per zawodniczka
    agg = df.groupby(['_norm_name','_team_id']).agg(
        avg_att   = ('_att_pts',  'mean'),
        avg_blocks= ('_blocks',   'mean'),
        avg_rec_t = ('_rec_tot',  'mean'),
        avg_rec_p = ('_rec_pos',  'mean'),
        avg_srv   = ('_srv_ace',  'mean'),
        is_libero = ('_is_libero','any'),
        n_matches = ('_att_pts',  'count'),
    ).reset_index()

    # OPP: najlepiej atakująca w każdej drużynie (top 1 per team)
    opp_ids = set()
    for tid, grp in agg[~agg['is_libero']].groupby('_team_id'):
        if not grp.empty:
            opp_row = grp.loc[grp['avg_att'].idxmax()]
            opp_ids.add((opp_row['_norm_name'], opp_row['_team_id']))

    def assign_position(row):
        if row['is_libero']:
            return 'L'
        key = (row['_norm_name'], row['_team_id'])
        if key in opp_ids:
            return 'OPP'
        if row['avg_blocks'] >= 0.7:
            return 'MB'
        if row['avg_att'] < 1.5 and row['avg_blocks'] < 0.5:
            return 'S'
        return 'OH'

    agg['inferred_position'] = agg.apply(assign_position, axis=1)

    log.info('Rozkład pozycji (heurystyka):')
    log.info(str(agg['inferred_position'].value_counts().to_dict()))

    return agg[['_norm_name','_team_id','inferred_position',
                'avg_att','avg_blocks','avg_rec_t','n_matches']]

# ── Uruchom heurystykę ─────────────────────────────────────────
position_heuristic_df = infer_positions_from_stats(players_raw_df)

# ── Nanieś pozycje na players_master ──────────────────────────
pos_lookup = position_heuristic_df.set_index(
    ['_norm_name','_team_id'])['inferred_position'].to_dict()

players_master_df['position'] = players_master_df.apply(
    lambda r: pos_lookup.get(
        (r['normalized_player_name'], r['team_id']),
        r['position']
    ), axis=1
)

# Zaktualizuj confidence (libero i znane pozycje → wyższy confidence)
def update_confidence(row):
    if row['position'] == 'UNKNOWN':
        return 'low'
    if row['position'] == 'L':
        return 'high'
    n = int(row['matches_count'])
    if n >= 5: return 'normal'
    if n >= 3: return 'medium'
    return 'low'

players_master_df['confidence_flag'] = players_master_df.apply(update_confidence, axis=1)

# ── Podsumowanie ───────────────────────────────────────────────
print('\n✅ Przypisano pozycje (heurystyka):')
print(players_master_df['position'].value_counts().to_string())
print(f'\nUNKNOWN pozostało: {(players_master_df["position"]=="UNKNOWN").sum()}')

print('\n📋 Podgląd (top 15 wg drużyny):')
print(players_master_df[['normalized_player_name','current_team_name',
                          'position','matches_count','confidence_flag']]
      .sort_values(['current_team_name','position']).head(15).to_string(index=False))

# ── Eksportuj CSV do ręcznej korekty ──────────────────────────
csv_path = '/content/positions_to_review.csv'
players_master_df[[
    'player_id','normalized_player_name','current_team_name',
    'position','matches_count','confidence_flag'
]].to_csv(csv_path, index=False)

from google.colab import files
files.download(csv_path)
print(f'\n📥 Pobrano: positions_to_review.csv')
print('   Popraw kolumnę "position" (S/OPP/OH/MB/L/UNKNOWN) i wgraj z powrotem.')

In [ ]:
# ═══════════════════════════════════════════════════════════════
# FAZA A2 — KROK 2: Naniesienie ręcznych korekt pozycji
# Korekty na podstawie weryfikacji z volleybox.net
# ═══════════════════════════════════════════════════════════════

MANUAL_CORRECTIONS = [
    ("De Almeida Rios Naiane",  "S"),
    ("Dautova Aida",            "OPP"),
    ("Rachkovska Vangeliya",    "OH"),
    ("Popova Alina",            "OPP"),
    ("Danard-Selosse Enora",    "S"),
    ("Starostenko Kristina",    "MB"),
    ("Gamanovich Valeriia",     "OPP"),
    ("De Zwart Laura",          "MB"),
    ("Medina Winderlys",        "OH"),
    ("Converty Emma",           "OH"),
]

before = (players_master_df["position"] == "UNKNOWN").sum()

for player_name, position in MANUAL_CORRECTIONS:
    # Szukaj po pierwszym członie nazwiska (bardziej odporne na różnice w zapisie)
    first_token = player_name.split()[0]
    mask = players_master_df["normalized_player_name"].str.contains(
        first_token, case=False, na=False
    )
    matched = players_master_df.loc[mask, "normalized_player_name"].tolist()
    if matched:
        players_master_df.loc[mask, "position"] = position
        print(f"  ✅ {matched[0]:35s} → {position}")
    else:
        print(f"  ⚠️  Nie znaleziono: {player_name}")

# Odśwież confidence
players_master_df["confidence_flag"] = players_master_df.apply(
    update_confidence, axis=1
)

after = (players_master_df["position"] == "UNKNOWN").sum()
print(f"\n📊 Rozkład pozycji po korektach:")
print(players_master_df["position"].value_counts().to_string())
print(f"\nUNKNOWN: {before} → {after}")

## 🅱️ FAZA B — Player Impact Score (PIS) per mecz

In [ ]:
# ═══════════════════════════════════════════════════════════════
# FAZA B — PLAYER MATCH FEATURES → player_match_features
# Jedna zawodniczka = jeden wiersz na mecz.
# Oblicza PIS v0 per mecz + wszystkie kolumny wymagane przez schemat.
# ═══════════════════════════════════════════════════════════════

def compute_starter_flag(pis_match: float, global_median: float) -> int:
    """
    starter_flag proxy v0:
    1 = zawodniczka zagrała z sensowną partycypacją (PIS >= mediana),
    0 = rotation/bench.
    W v1 zastąp prawdziwym lineup z protokołu.
    """
    return 1 if pis_match >= global_median else 0

def build_player_match_features(
    players_raw:    pd.DataFrame,
    players_master: pd.DataFrame,
    matches_df:     pd.DataFrame,
) -> pd.DataFrame:
    """
    Faza B: Buduje player_match_features.
    Kolumny zgodne ze schematem zakładki Google Sheets.
    PIS v0 = attack*w1 + blocks*w2 + ace*w3 - errors*w4
    """
    if players_raw.empty:
        return pd.DataFrame()

    cols = list(players_raw.columns)

    def col(kw):
        return next((c for c in cols if kw in c.lower()), None)

    c_name    = col("player_name")
    c_team    = col("team_name")
    c_match   = col("match_id")
    c_libero  = col("libero")
    c_mdt     = col("datetime")
    c_att_pts = col("att_pts")
    c_blocks  = col("blocks")
    c_srv_ace = col("srv_ace")
    c_srv_err = col("srv_err")
    c_att_err = col("att_err")
    c_rec_tot = col("rec_tot")
    c_rec_pos = col("rec_pos")

    df = players_raw.copy()
    df["_norm_name"] = df[c_name].fillna("").apply(normalize_name)
    df["_norm_team"] = df[c_team].fillna("").apply(normalize_name)
    df["_canonical_team"] = df[c_team].fillna("").apply(get_canonical_team_name)
    df["_team_id"]   = df["_canonical_team"].apply(make_team_id)
    df["_match_id"]  = df[c_match].fillna("").astype(str)
    df["_match_date"]= df[c_mdt].fillna("").astype(str) if c_mdt else ""
    df["_is_libero"] = (
        df[c_libero].fillna("").astype(str).str.upper()
        .isin(["L","YES","TRUE","1","LIBERO"])
    ) if c_libero else False

    # Statystyki numeryczne
    df["_att_pts"] = df[c_att_pts].apply(safe_float) if c_att_pts else 0.0
    df["_blocks"]  = df[c_blocks].apply(safe_float)  if c_blocks  else 0.0
    df["_srv_ace"] = df[c_srv_ace].apply(safe_float) if c_srv_ace else 0.0
    df["_srv_err"] = df[c_srv_err].apply(safe_float) if c_srv_err else 0.0
    df["_att_err"] = df[c_att_err].apply(safe_float) if c_att_err else 0.0
    df["_rec_tot"] = df[c_rec_tot].apply(safe_float) if c_rec_tot else 0.0
    df["_rec_pos"] = df[c_rec_pos].apply(safe_float) if c_rec_pos else 0.0

    # Lookup player_id i position z players_master
    # Priorytet 1: (name, team_id) — exact match
    pid_lookup_exact = players_master.set_index(
        ["normalized_player_name","team_id"]
    )["player_id"].to_dict()
    pos_lookup_exact = players_master.set_index(
        ["normalized_player_name","team_id"]
    )["position"].to_dict()

    # Priorytet 2: tylko po nazwisku (fallback gdy team_id się różni między fazami)
    pid_lookup_name = players_master.drop_duplicates("normalized_player_name").set_index(
        "normalized_player_name"
    )["player_id"].to_dict()
    pos_lookup_name = players_master.drop_duplicates("normalized_player_name").set_index(
        "normalized_player_name"
    )["position"].to_dict()

    def resolve_pid(row):
        key_exact = (row["_norm_name"], row["_team_id"])
        # Try exact (name + team_id) first
        pid = pid_lookup_exact.get(key_exact)
        if pid: return pid
        # Fallback: name only (handles team_id hash mismatch)
        return pid_lookup_name.get(row["_norm_name"],
               make_player_id(row["_norm_name"], row["_team_id"]))

    def resolve_pos(row):
        if row["_is_libero"]:
            return "L"
        # Try exact (name + team_id)
        key_exact = (row["_norm_name"], row["_team_id"])
        pos = pos_lookup_exact.get(key_exact)
        if pos: return pos
        # Fallback: name only
        pos = pos_lookup_name.get(row["_norm_name"])
        if pos: return pos
        return "UNKNOWN"

    df["player_id"] = df.apply(resolve_pid, axis=1)
    df["position"]  = df.apply(resolve_pos, axis=1)

    # ── PIS v0 ────────────────────────────────────────────────
    df["pis_base"] = df.apply(lambda r: round(
        r["_att_pts"] * PIS_COEFF["att_pts"]
        + r["_blocks"]  * PIS_COEFF["blocks"]
        + r["_srv_ace"] * PIS_COEFF["srv_ace"]
        + r["_srv_err"] * PIS_COEFF["srv_err"]   # koeff ujemny
        + r["_att_err"] * PIS_COEFF["att_err"],
        2
    ), axis=1)

    # attack_eff = att_pts / (att_pts + att_err) jeśli mianownik > 0
    df["attack_eff"] = df.apply(lambda r: round(
        r["_att_pts"] / (r["_att_pts"] + r["_att_err"])
        if (r["_att_pts"] + r["_att_err"]) > 0 else 0.0,
        4
    ), axis=1)

    # pis_match = pis_base + bonus jakościowy
    def pis_match_full(row):
        base = row["pis_base"]
        # Bonus efektywności ataku (powyżej 20% skuteczności)
        eff_bonus = max(0.0, (row["attack_eff"] - 0.20)) * 8
        # Reception bonus dla OH i L
        rec_bonus = 0.0
        if row["position"] in ("OH", "L") and row["_rec_tot"] > 0:
            rec_rate = row["_rec_pos"] / row["_rec_tot"]
            rec_bonus = max(0.0, (rec_rate - 0.45)) * 10
        return round(base + eff_bonus + rec_bonus, 2)

    df["pis_match"] = df.apply(pis_match_full, axis=1)

    # starter_flag proxy (globalnie per mecz)
    global_median = df["pis_match"].median()
    df["starter_flag"] = df["pis_match"].apply(
        lambda x: compute_starter_flag(x, global_median)
    )

    # role i role_weight
    df["role"] = df["starter_flag"].apply(
        lambda x: "starter" if x == 1 else "rotation"
    )
    df["role_weight"] = df["role"].apply(
        lambda r: ROLE_WEIGHTS.get("S", 1.0) if r == "starter" else 0.70
    )

    # Właściwy role_weight wg pozycji
    df["role_weight"] = df["position"].apply(
        lambda p: ROLE_WEIGHTS.get(p, ROLE_WEIGHTS["UNKNOWN"])
    )

    # tis_contribution = pis_match * role_weight
    df["tis_contribution"] = (df["pis_match"] * df["role_weight"]).round(2)

    # ── Lookup match_date z matches_df ────────────────────────
    match_date_lookup = {}
    if not matches_df.empty:
        m_cols = list(matches_df.columns)
        c_mid  = next((c for c in m_cols if "match_id" in c.lower()), None)
        c_mdt2 = next((c for c in m_cols if "datetime" in c.lower()), None)
        if c_mid and c_mdt2:
            match_date_lookup = matches_df.set_index(c_mid)[c_mdt2].to_dict()

    df["match_date"] = df["_match_id"].apply(
        lambda mid: match_date_lookup.get(mid, df.loc[df["_match_id"]==mid, "_match_date"].iloc[0]
        if not df.loc[df["_match_id"]==mid, "_match_date"].empty else "")
    )

    # ── Złóż finalny DataFrame ────────────────────────────────
    result = df[[
        "_match_id", "match_date", "_team_id", "player_id",
        "position", "starter_flag", "role", "role_weight",
        "attack_eff", "pis_base", "pis_match", "tis_contribution"
    ]].copy()

    result.columns = [
        "match_id", "match_date", "team_id", "player_id",
        "position", "starter_flag", "role", "role_weight",
        "attack_eff", "pis_base", "pis_match", "tis_contribution"
    ]

    log.info(
        f"player_match_features: {len(result)} wierszy | "
        f"avg PIS={result['pis_match'].mean():.3f} | "
        f"starter: {result['starter_flag'].sum()} / {len(result)}"
    )
    return result

# ── Uruchom ────────────────────────────────────────────────────
player_match_features_df = build_player_match_features(
    players_raw_df, players_master_df, matches_df
)

# Zachowaj też jako pis_per_match_df (wymagane przez Fazy C–E)
pis_per_match_df = player_match_features_df[[
    "player_id", "match_id", "pis_match"
]].copy()
pis_per_match_df["normalized_name"] = player_match_features_df["player_id"].apply(
    lambda pid: players_master_df.loc[
        players_master_df["player_id"]==pid, "normalized_player_name"
    ].iloc[0] if pid in players_master_df["player_id"].values else pid
)
pis_per_match_df["team_id"]   = player_match_features_df["team_id"]
pis_per_match_df["is_libero"] = player_match_features_df["position"] == "L"

print(f"\n✅ Faza B zakończona:")
print(f"   player_match_features: {len(player_match_features_df)} wierszy")
print(f"   avg pis_match:         {player_match_features_df['pis_match'].mean():.3f}")
print(f"   avg tis_contribution:  {player_match_features_df['tis_contribution'].mean():.3f}")
print(f"\n📋 Top 10 wg pis_match:")
print(player_match_features_df[[
    "match_id","team_id","player_id","position",
    "role","pis_base","pis_match","tis_contribution"
]].sort_values("pis_match", ascending=False).head(10).to_string(index=False))

## 🅲 FAZA C — Rolling Baseline → `player_pis_rolling`

In [ ]:
# ═══════════════════════════════════════════════════════════════
# FAZA C — PLAYER ROLLING → player_rolling
# Buduje formę zawodniczki: pis_last_3, pis_last_5, pis_season, pis_rolling.
# Schemat zgodny z zakładką player_rolling w Google Sheets.
# ═══════════════════════════════════════════════════════════════

def build_player_rolling(
    player_match_features: pd.DataFrame,
    players_master:        pd.DataFrame,
) -> pd.DataFrame:
    """
    Faza C: Rolling baseline dla każdej zawodniczki.

    Formuły zgodne ze schematem Sheets:
      pis_last_3  = średnia z 3 ostatnich meczów (wg match_id malejąco)
      pis_last_5  = średnia z 5 ostatnich meczów
      pis_season  = średnia z całego sezonu
      pis_rolling = pis_last_3*0.20 + pis_last_5*0.30 + pis_season*0.50
    """
    if player_match_features.empty:
        return pd.DataFrame()

    # Lookup z players_master
    master_idx = players_master.set_index("player_id")

    records = []

    for player_id, grp in player_match_features.groupby("player_id"):

        # Sortuj wg match_id malejąco (najnowsze pierwsze) — jak SORT(...,FALSE) w Sheets
        grp_sorted = grp.sort_values("match_id", ascending=False).reset_index(drop=True)
        pis_vals   = grp_sorted["pis_match"].tolist()
        n          = len(pis_vals)

        pis_season = round(float(np.mean(pis_vals)), 2)          if n >= 1 else 0.0
        pis_last_5 = round(float(np.mean(pis_vals[:5])), 2)      if n >= 1 else 0.0
        pis_last_3 = round(float(np.mean(pis_vals[:3])), 2)      if n >= 1 else 0.0

        # pis_rolling = last_3*0.20 + last_5*0.30 + season*0.50
        pis_rolling = round(
            pis_last_3 * PIS_W_LAST3
            + pis_last_5 * PIS_W_LAST5
            + pis_season * PIS_W_SEASON,
            2
        )

        # ── start_rate: jak często zawodniczka gra set 1 ────────
        # Liczymy ile meczów grała (set_no == 1 lub po prostu unikalnych match_id)
        matches_played = int(grp_sorted["match_id"].astype(str).str.strip().nunique())
        # start_rate: v0 = 1.0 (każdy mecz = start, bo player_match_features jest per mecz)
        # W v1 zastąp prawdziwym set_no z players_raw
        start_rate = 1.0  # placeholder — wszystkie rekordy to starterzy w tej wersji

        # ── form_factor: stosunek formy ostatnich 3 do średniej sezonu ─
        if pis_season > 0:
            form_factor = round(min(1.3, pis_last_3 / pis_season), 3)
        else:
            form_factor = 1.0

        # ── first_match_id: do wykrywania nowych nabytków ────────
        first_match_id = int(grp_sorted["match_id"].min()) if n > 0 else 0
        last_match_id  = int(grp_sorted["match_id"].max()) if n > 0 else 0

        # ── gap_detected: przerwa ≥ 3 mecze (kontuzja/powrót) ───
        sorted_mids = sorted(grp_sorted["match_id"].astype(int).tolist())
        gap_detected = False
        if len(sorted_mids) >= 4:
            # Sprawdź czy jest przerwa ≥ 3 kolejnych match_id
            for j in range(1, len(sorted_mids)):
                if sorted_mids[j] - sorted_mids[j-1] > 3:
                    gap_detected = True
                    break

        # Lookup z players_master
        if player_id in master_idx.index:
            r         = master_idx.loc[player_id]
            team_id   = r["team_id"]
            player_name = r["normalized_player_name"]
            position  = r["position"]
        else:
            team_id     = grp_sorted["team_id"].iloc[0]
            player_name = player_id
            position    = grp_sorted["position"].iloc[0] if "position" in grp_sorted.columns else "UNKNOWN"

        records.append({
            "player_id":   player_id,
            "team_id":     team_id,
            "team_name":   get_canonical_name_by_tid(team_id),
            "player_name": player_name,
            "position":    position,
            "pis_last_3":  pis_last_3,
            "pis_last_5":  pis_last_5,
            "pis_season":  pis_season,
            "pis_rolling":    pis_rolling,
            "start_rate":     start_rate,
            "form_factor":    form_factor,
            "matches_played": matches_played,
            "first_match_id": first_match_id,
            "last_match_id":  last_match_id,
            "gap_detected":   gap_detected,
        })

    df = pd.DataFrame(records)
    log.info(
        f"player_rolling: {len(df)} zawodniczek | "
        f"avg pis_rolling={df['pis_rolling'].mean():.3f}"
    )
    return df

# ── Uruchom ────────────────────────────────────────────────────
player_rolling_df = build_player_rolling(player_match_features_df, players_master_df)

# Zachowaj alias dla kompatybilności z Fazami D–E (używają pis_rolling_df)
pis_rolling_df = player_rolling_df.copy()

# Dodaj kolumny których potrzebują Fazy D/E a nie ma ich w nowym schemacie
# matches_played — liczba unikalnych meczów per zawodniczka
# Zawsze przeliczaj na nowo (ignoruj ewentualną kolumnę z wcześniejszego błędu)
_mp_map = (
    player_match_features_df
    .assign(_mid=lambda d: d["match_id"].astype(str).str.strip())
    .groupby("player_id")["_mid"]
    .nunique()
)
pis_rolling_df["matches_played"] = (
    pis_rolling_df["player_id"]
    .map(_mp_map)
    .fillna(0)
    .astype(int)
)
print(f"   matches_played: min={pis_rolling_df['matches_played'].min()}, "
      f"max={pis_rolling_df['matches_played'].max()}, "
      f"median={pis_rolling_df['matches_played'].median():.0f}")

if "recent_starts_proxy" not in pis_rolling_df.columns:
    global_median = player_match_features_df["pis_match"].median()
    starts_lookup = (
        player_match_features_df
        .sort_values("match_id", ascending=False)
        .groupby("player_id")
        .head(5)
        .groupby("player_id")["starter_flag"]
        .sum()
    )
    pis_rolling_df["recent_starts_proxy"] = pis_rolling_df["player_id"].map(
        starts_lookup
    ).fillna(0).astype(int)

if "role_weight" not in pis_rolling_df.columns:
    pis_rolling_df["role_weight"] = pis_rolling_df["position"].apply(
        lambda p: ROLE_WEIGHTS.get(p, ROLE_WEIGHTS["UNKNOWN"])
    )

if "confidence_flag" not in pis_rolling_df.columns:
    def conf(n):
        if n <= 2:  return "low"
        if n <= 4:  return "medium"
        return "normal"
    pis_rolling_df["confidence_flag"] = pis_rolling_df["matches_played"].apply(conf)

print(f"\n✅ Faza C zakończona:")
print(f"   player_rolling: {len(player_rolling_df)} zawodniczek")
print(f"\n📋 Top 10 wg pis_rolling:")
print(player_rolling_df[["player_name","team_id","position","pis_last_3","pis_last_5","pis_season","pis_rolling"]]
      .sort_values("pis_rolling", ascending=False)
      .head(10)
      .to_string(index=False))

## 🅳 FAZA D — Typical Lineups → `expected_lineups`

In [ ]:
# ═══════════════════════════════════════════════════════════════
# FAZA D — Expected Lineups → `expected_lineups`
#
# Buduje oczekiwany skład startowy per drużyna używając:
#   TIS_expected_slot = pis_rolling × role_weight × start_rate × form_factor
#
# Hierarchia wyboru zawodniczki per slot (malejąco):
#   1. start_rate > 0.7  (regularna w wyjściowej szóstce)
#   2. form_factor > 0.9 (dobra forma)
#   3. pis_rolling       (historyczna jakość)
#   4. Fallback: najlepsza nieprzypisana z drużyny
# ═══════════════════════════════════════════════════════════════

def build_expected_lineups(pis_rolling: pd.DataFrame,
                           players_master: Optional[pd.DataFrame] = None) -> pd.DataFrame:
    """
    Faza D: Expected lineup per drużyna.
    Każda drużyna → 7 slotów: S, OPP, OH1, OH2, MB1, MB2, L
    """
    if pis_rolling.empty:
        return pd.DataFrame()

    today = date.today().isoformat()

    # Dołącz team_name z cache teams_master (kolumna C = team_name_canonical)
    df_work = pis_rolling.copy()

    # Upewnij się że cache jest załadowany
    if not _TEAMS_MASTER_CACHE:
        load_teams_master_cache(BOOK)

    # Mapuj team_id → canonical name (cache + hardcoded fallback)
    df_work["team_name"] = df_work["team_id"].apply(get_canonical_name_by_tid)

    # Weryfikacja — żaden hash nie powinien zostać
    hashes_left = df_work[df_work["team_name"].str.match(r"^T_[0-9A-F]{8}$", na=False)]
    if not hashes_left.empty:
        print(f"   ⚠️  Nieznane team_id (dodaj do teams_master): {hashes_left['team_id'].unique().tolist()}")
    else:
        print(f"   ✅ Wszystkie team_id rozwiązane: {sorted(df_work['team_name'].unique())}")

    # Dołącz normalized_player_name jeśli brak
    if "normalized_player_name" not in df_work.columns:
        if "player_name" in df_work.columns:
            df_work["normalized_player_name"] = df_work["player_name"]
        else:
            df_work["normalized_player_name"] = df_work["player_id"]

    # Mapowanie slot → akceptowane pozycje
    SLOT_POSITIONS = {
        "S":   ["S"],
        "OPP": ["OPP"],
        "OH1": ["OH"],
        "OH2": ["OH"],
        "MB1": ["MB"],
        "MB2": ["MB"],
        "L":   ["L"],
    }
    SLOTS = list(SLOT_POSITIONS.keys())

    records = []

    for team_id, team_df in df_work.groupby("team_id"):
        team_name    = team_df["team_name"].iloc[0]
        assigned_ids = set()

        # Wzbogać dane o start_rate i form_factor (z player_rolling)
        def get_sr(row):
            return float(row.get("start_rate", 1.0) or 1.0)
        def get_ff(row):
            return float(row.get("form_factor", 1.0) or 1.0)

        for slot_name in SLOTS:
            accepted_pos = SLOT_POSITIONS[slot_name]

            # Kandydatki: właściwa pozycja, nieprzypisana, pis > 0
            candidates = team_df[
                team_df["position"].isin(accepted_pos) &
                ~team_df["player_id"].isin(assigned_ids) &
                (team_df["pis_rolling"] > 0)
            ].copy()

            # Score = pis_rolling × role_weight × start_rate × form_factor
            if not candidates.empty:
                candidates["_score"] = candidates.apply(
                    lambda r: float(r["pis_rolling"])
                              * ROLE_WEIGHTS.get(r["position"], ROLE_WEIGHTS["UNKNOWN"])
                              * float(r.get("start_rate", 1.0) or 1.0)
                              * float(r.get("form_factor", 1.0) or 1.0),
                    axis=1
                )
                candidates = candidates.sort_values("_score", ascending=False)

            # Fallback 1: dowolna nieprzypisana zawodniczka z drużyny
            if candidates.empty:
                candidates = team_df[
                    ~team_df["player_id"].isin(assigned_ids) &
                    (team_df["pis_rolling"] > 0)
                ].copy()
                if not candidates.empty:
                    candidates["_score"] = candidates["pis_rolling"]
                    candidates = candidates.sort_values("_score", ascending=False)

            # Fallback 2: jakakolwiek zawodniczka (nawet już przypisana)
            if candidates.empty:
                candidates = team_df[team_df["pis_rolling"] > 0].copy()
                if not candidates.empty:
                    candidates["_score"] = candidates["pis_rolling"]
                    candidates = candidates.sort_values("_score", ascending=False)

            if candidates.empty:
                continue

            best      = candidates.iloc[0]
            sr        = float(best.get("start_rate", 1.0) or 1.0)
            ff        = float(best.get("form_factor", 1.0) or 1.0)
            rw        = ROLE_WEIGHTS.get(best["position"], ROLE_WEIGHTS["UNKNOWN"])
            slot_val  = round(float(best["pis_rolling"]) * rw * sr * ff, 4)
            is_fb     = best["position"] not in accepted_pos or best["player_id"] in assigned_ids

            # Flagi kontekstowe
            gap     = bool(best.get("gap_detected", False))
            matches = int(best.get("matches_played", 0))
            # Nowy nabytek: < 4 mecze w sezonie lub first_match_id > 9050 (~Feb 2026)
            is_new  = matches <= 4 or int(best.get("first_match_id", 0)) > 9050

            flags = []
            if sr < 0.4:     flags.append("rotacja")
            if ff < 0.7:     flags.append("słaba_forma")
            if ff > 1.1:     flags.append("dobra_forma")
            if gap:          flags.append("powrot_po_przerwie")
            if is_new:       flags.append("nowy_nabytek")
            if is_fb:        flags.append(f"fallback_{best['position']}")

            assigned_ids.add(best["player_id"])

            records.append({
                "team_id":           team_id,
                "team_name":         team_name,
                "snapshot_date":     today,
                "slot_name":         slot_name,
                "player_id":         best["player_id"],
                "player_name":       best["normalized_player_name"],
                "position":          best["position"],
                "pis_rolling":       round(float(best["pis_rolling"]), 4),
                "start_rate":        round(sr, 3),
                "form_factor":       round(ff, 3),
                "selection_score":   round(float(best.get("_score", 0)), 4),
                "role_weight":       rw,
                "slot_value":        slot_val,
                "matches_played":    matches,
                "gap_detected":      gap,
                "is_new_signing":    is_new,
                "lineup_confidence": "low" if is_fb else ("high" if sr >= 0.7 else "medium"),
                "notes":             "|".join(flags) if flags else "regular",
            })

    df = pd.DataFrame(records)
    if df.empty:
        return df

    # Sortuj
    SLOT_ORDER = {s: i for i, s in enumerate(SLOTS)}
    df["_sk"] = df["slot_name"].map(SLOT_ORDER).fillna(99).astype(int)
    df = df.sort_values(["team_name", "_sk"]).drop(columns=["_sk"])

    return df.reset_index(drop=True)

# ── Uruchom Fazę D ────────────────────────────────────────────
print("🅳 Budowanie Expected Lineups...")
expected_lineups_df = build_expected_lineups(pis_rolling_df, players_master_df)

print(f"   Wierszy: {len(expected_lineups_df)}")
print(f"   Drużyn:  {expected_lineups_df['team_name'].nunique()}")
print()
print("📋 Expected Lineups — przegląd per drużyna:")
print("=" * 90)
for tname, grp in expected_lineups_df.groupby("team_name"):
    tis = grp["slot_value"].astype(float).sum()
    print(f"\n🏐 {tname} | TIS_expected = {tis:.2f}")
    print(f"   {'Slot':5s} | {'Zawodniczka':28s} | {'Poz':4s} | {'PIS':6s} | {'SR':5s} | {'FF':5s} | {'Slot_Val':8s} | Flagi")
    print(f"   {'-'*5}-+-{'-'*28}-+-{'-'*4}-+-{'-'*6}-+-{'-'*5}-+-{'-'*5}-+-{'-'*8}-+-{'-'*20}")
    for _, r in grp.iterrows():
        conf_icon = "⭐" if r["lineup_confidence"] == "high" else ("🔶" if r["lineup_confidence"] == "medium" else "🔴")
        print(f"   {r['slot_name']:5s} | {r['player_name']:28s} | {r['position']:4s} | "
              f"{float(r['pis_rolling']):6.2f} | {float(r['start_rate']):5.2f} | "
              f"{float(r['form_factor']):5.2f} | {float(r['slot_value']):8.4f} | "
              f"{conf_icon} {r['notes']}")
print("\n" + "=" * 90)

# Alias dla zgodności z resztą kodu
typical_lineups_df = expected_lineups_df
print("\n✅ Faza D gotowa — expected_lineups_df zbudowany")


## 🅴 FAZA E — Team Baselines → `team_baselines`

In [ ]:
def build_team_baselines(expected_lineups: pd.DataFrame) -> pd.DataFrame:
    """
    Faza E: Liczy siłę drużyny na podstawie expected lineup.
    lineup_confidence values: 'high', 'medium', 'low' (z Fazy D)
    """
    if expected_lineups.empty:
        return pd.DataFrame()

    today = date.today().isoformat()
    records = []

    for team_id, grp in expected_lineups.groupby('team_id'):
        # Canonical team name — nigdy nie używaj UNKNOWN
        raw_name = grp['team_name'].iloc[0]
        team_name = get_canonical_name_by_tid(team_id)
        if not team_name or team_name == team_id:
            team_name = get_canonical_team_name(raw_name)
        if not team_name or team_name == 'UNKNOWN':
            team_name = raw_name  # ostateczny fallback

        slot_values = grp['slot_value'].astype(float).tolist()
        tis_typical = sum(slot_values)

        # Top-3 share (dominacja wierchuszki)
        sorted_vals = sorted(slot_values, reverse=True)
        top3_sum    = sum(sorted_vals[:3])
        top3_share  = round((top3_sum / tis_typical), 4) if tis_typical > 0 else 0.0

        # Depth score: siła rotacji (sloty 4-7 / tis)
        depth_score = round(((tis_typical - top3_sum) / tis_typical), 4) if tis_typical > 0 else 0.0

        # Lineup stability: % slotów z confidence 'high' lub 'medium'
        # Faza D używa: 'high' (start_rate>=0.7), 'medium', 'low' (fallback)
        stable = grp[grp['lineup_confidence'].isin(['high', 'medium'])].shape[0]
        lineup_stability_score = round(stable / max(len(grp), 1), 4)

        # start_rate średnia dla drużyny (jeśli kolumna dostępna)
        avg_start_rate = 0.0
        if 'start_rate' in grp.columns:
            avg_start_rate = round(float(grp['start_rate'].astype(float).mean()), 3)

        # form_factor średnia
        avg_form_factor = 1.0
        if 'form_factor' in grp.columns:
            avg_form_factor = round(float(grp['form_factor'].astype(float).mean()), 3)

        # Composite baseline strength index (0–1)
        # Benchmark: 7 zawodniczek × avg PIS 5.0 × avg role_weight 1.1 = 38.5
        # Dla LNV Women kalibracja: realny TIS ~50-85 → benchmark ~65
        BENCHMARK_TIS = 65.0
        baseline_strength_index = round(min(tis_typical / BENCHMARK_TIS, 1.0), 4)

        # Team confidence
        low_conf = grp[grp['lineup_confidence'] == 'low'].shape[0]
        team_conf = 'low' if low_conf >= 4 else ('medium' if low_conf >= 2 else 'high')

        # Gap/injury flags
        has_gap = False
        is_new_signing = False
        if 'gap_detected' in grp.columns:
            has_gap = bool(grp['gap_detected'].astype(str).str.lower().isin(['true','1']).any())
        if 'is_new_signing' in grp.columns:
            is_new_signing = bool(grp['is_new_signing'].astype(str).str.lower().isin(['true','1']).any())

        notes_parts = []
        if has_gap:       notes_parts.append('powrot_po_przerwie')
        if is_new_signing:notes_parts.append('nowy_nabytek')

        records.append({
            'team_id':                 team_id,
            'team_name':               team_name,
            'snapshot_date':           today,
            'tis_expected':            round(tis_typical, 4),
            'top_3_share':             top3_share,
            'depth_score':             depth_score,
            'lineup_stability_score':  lineup_stability_score,
            'avg_start_rate':          avg_start_rate,
            'avg_form_factor':         avg_form_factor,
            'baseline_strength_index': baseline_strength_index,
            'confidence_flag':         team_conf,
            'notes':                   ' | '.join(notes_parts),
        })

    df = pd.DataFrame(records)
    log.info(f'team_baselines: {len(df)} drużyn')
    return df

team_baselines_df = build_team_baselines(expected_lineups_df)

print(f'\n✅ Faza E zakończona: {len(team_baselines_df)} drużyn')
print(team_baselines_df[['team_name','tis_expected','top_3_share','lineup_stability_score','baseline_strength_index','confidence_flag']]
      .sort_values('tis_expected', ascending=False).to_string(index=False))

In [ ]:
# ══════════════════════════════════════════════════════
# DEBUG — znajdź nazwy drużyn dla nieznanych team_id
# Uruchom jeśli widzisz WARNING: Nieznany team_id
# ══════════════════════════════════════════════════════
import hashlib, unicodedata, re as _re

def _norm(s):
    n = _re.sub(r"\s+"," ",str(s).strip())
    n = unicodedata.normalize("NFD",n)
    return "".join(c for c in n if unicodedata.category(c)!="Mn").title().strip()

def _tid(name): return "T_"+hashlib.md5(_norm(name).encode()).hexdigest()[:8].upper()

# Wczytaj players_raw i sprawdź wszystkie unikalne nazwy drużyn
ws_pr = BOOK.worksheet("players_raw")
pr_data = ws_pr.get_all_values()
pr_hdrs = pr_data[0]
c_team = next(i for i,h in enumerate(pr_hdrs) if 'team' in h.lower())

raw_names = sorted(set(str(r[c_team]).strip() for r in pr_data[1:] if len(r)>c_team and r[c_team].strip()))
print(f"Unikalne nazwy drużyn w players_raw ({len(raw_names)}):")
for name in raw_names:
    t = _tid(name)
    canonical = get_canonical_team_name(name)
    ct = _tid(canonical)
    in_fallback = t in _TID_FALLBACK
    marker = "✅" if in_fallback else "❌ BRAK W FALLBACK"
    print(f"  {marker} '{name}' → tid={t} → canonical='{canonical}' → canon_tid={ct}")


## 🅵 FAZA F — Live Lineups: schema + mock loader

In [ ]:
import pandas as pd
import numpy as np

LIVE_LINEUP_HEADERS = [
    'match_id', 'team_id', 'team_name',
    'source_type', 'source_timestamp',
    'set_no', 'slot_name',
    'live_player_name_raw', 'live_player_name_normalized',
    'resolved_player_id', 'resolved_position',
    'resolution_confidence', 'is_missing_from_typical', 'notes'
]

def normalize_live_name(raw: str) -> str:
    return normalize_name(raw)

def resolve_live_player(
    live_name_normalized: str,
    team_id: str,
    players_master: pd.DataFrame,
    fuzzy_threshold: int = 82
) -> Tuple[str, str, str]:
    """
    Rozwiązuje live player name → player_id.
    Zwraca (player_id, position, resolution_confidence).
    """
    team_players = players_master[players_master['team_id'] == team_id]

    if team_players.empty:
        return make_player_id(live_name_normalized, team_id), 'UNKNOWN', 'low'

    # Exact match
    exact = team_players[team_players['normalized_player_name'] == live_name_normalized]
    if not exact.empty:
        r = exact.iloc[0]
        return r['player_id'], r['position'], 'high'

    # Fuzzy match
    names = team_players['normalized_player_name'].tolist()
    result = process.extractOne(
        live_name_normalized, names,
        scorer=fuzz.token_sort_ratio,
        score_cutoff=fuzzy_threshold
    )
    if result:
        matched_name = result[0]
        r = team_players[team_players['normalized_player_name'] == matched_name].iloc[0]
        return r['player_id'], r['position'], 'medium'

    # Nierozwiązana
    return make_player_id(live_name_normalized, team_id), 'UNKNOWN', 'low'

def load_live_lineup_from_dict(
    raw_data: List[Dict],
    players_master: pd.DataFrame,
    expected_lineups: pd.DataFrame
) -> pd.DataFrame:
    """
    Parser dla live lineup data (dict/JSON format).
    raw_data: lista słowników z polami match_id, team_id, team_name,
              slot_name, live_player_name_raw, set_no, source_type.
    """
    records = []
    now = datetime.now().strftime('%Y-%m-%d %H:%M:%S')

    # Zbiór player_id w typical lineup per team
    typical_ids: Dict[str, set] = {}
    for team_id, grp in expected_lineups.groupby('team_id'):
        typical_ids[team_id] = set(grp['player_id'].tolist())

    for entry in raw_data:
        match_id       = str(entry.get('match_id', ''))
        team_id        = str(entry.get('team_id', ''))
        team_name      = str(entry.get('team_name', ''))
        slot_name      = str(entry.get('slot_name', ''))
        raw_name       = str(entry.get('live_player_name_raw', ''))
        set_no         = str(entry.get('set_no', '1'))
        source_type    = str(entry.get('source_type', 'mock'))

        norm_name = normalize_live_name(raw_name)
        player_id, position, conf = resolve_live_player(
            norm_name, team_id, players_master
        )

        is_missing = (
            team_id in typical_ids
            and player_id not in typical_ids[team_id]
        )

        records.append({
            'match_id':                  match_id,
            'team_id':                   team_id,
            'team_name':                 team_name,
            'source_type':               source_type,
            'source_timestamp':          now,
            'set_no':                    set_no,
            'slot_name':                 slot_name,
            'live_player_name_raw':      raw_name,
            'live_player_name_normalized': norm_name,
            'resolved_player_id':        player_id,
            'resolved_position':         position,
            'resolution_confidence':     conf,
            'is_missing_from_typical':   str(is_missing),
            'notes':                     '',
        })

    return pd.DataFrame(records)

print('✅ Faza F: Schemat i parser live lineups gotowy')
print(f'   Pola live_lineups: {LIVE_LINEUP_HEADERS}')

In [ ]:
# ── Mock data dla testów (zastąp prawdziwymi danymi gdy dostępne) ──
# Pobieramy prawdziwe team_id z naszego players_master

def build_mock_live_lineups(
    players_master: pd.DataFrame,
    expected_lineups: pd.DataFrame,
    n_matches: int = 2
) -> List[Dict]:
    """
    Buduje mock live lineup na bazie prawdziwych danych z players_master.
    Dla jednego meczu: symuluje brak kluczowej zawodniczki (shock scenario).
    """
    if players_master.empty or expected_lineups.empty:
        return []

    mock_entries = []
    teams = expected_lineups['team_id'].unique()[:2]   # 2 drużyny

    for i, team_id in enumerate(teams):
        team_lineups = expected_lineups[expected_lineups['team_id'] == team_id]
        team_name    = team_lineups['team_name'].iloc[0]
        match_id     = f'MOCK_{9000 + i}'

        for j, (_, slot_row) in enumerate(team_lineups.iterrows()):
            # Dla meczu #2: pomiń najlepszą zawodniczkę (lineup shock)
            if i == 1 and j == 0:
                # Zastąp najlepszą zawodniczkę rezerwową (nieco inne nazwisko)
                raw_name = slot_row['player_name'] + ' Jr'   # symulacja innego gracza
            else:
                raw_name = slot_row['player_name']

            mock_entries.append({
                'match_id':           match_id,
                'team_id':            team_id,
                'team_name':          team_name,
                'slot_name':          slot_row['slot_name'],
                'live_player_name_raw': raw_name,
                'set_no':             '1',
                'source_type':        'mock',
            })

    return mock_entries

mock_raw = build_mock_live_lineups(players_master_df, expected_lineups_df)
live_lineups_df = load_live_lineup_from_dict(mock_raw, players_master_df, expected_lineups_df)

print(f'✅ Mock live lineup: {len(live_lineups_df)} wierszy')
print(live_lineups_df[['match_id','team_name','slot_name','live_player_name_normalized',
                        'resolved_position','resolution_confidence','is_missing_from_typical']].head(14))

## 🔴 LIVE LINEUP — Mecz 9086: Terville Florange vs Béziers (14.03.2026)

In [ ]:
import pandas as pd
import numpy as np
import unicodedata, hashlib, re
from datetime import datetime
from rapidfuzz import process as rfp, fuzz as rff

# ── Inline helpers (na wypadek uruchomienia bez poprzednich komórek) ─
def normalize_name(raw):
    if not raw or not isinstance(raw, str): return ""
    name = re.sub(r"\s+", " ", raw.strip())
    nfd  = unicodedata.normalize("NFD", name)
    return "".join(c for c in nfd if unicodedata.category(c) != "Mn").title().strip()

def make_team_id(team_name):
    key = normalize_name(team_name)
    return "T_" + hashlib.md5(key.encode()).hexdigest()[:8].upper()

def make_player_id(normalized_name, team_id):
    key = f"{normalized_name}|{team_id}"
    return "P_" + hashlib.md5(key.encode()).hexdigest()[:10].upper()

def safe_float(val, default=0.0):
    try:
        f = float(str(val).strip().replace(",", "."))
        return f if np.isfinite(f) else default
    except: return default

def ensure_worksheet(book, name, rows=500, cols=30):
    try: return book.worksheet(name)
    except Exception:
        return book.add_worksheet(title=name, rows=rows, cols=cols)

def save_df(book, sheet_name, df):
    if df is None or df.empty:
        print(f"  ⚠️  POMINIĘTO {sheet_name} — pusty DataFrame")
        return False
    try:
        import time, gspread
        ws = ensure_worksheet(book, sheet_name, rows=max(500, len(df)+10), cols=len(df.columns)+2)
        ws.clear(); time.sleep(1.2)
        ws.update("A1", [list(df.columns)], value_input_option="USER_ENTERED"); time.sleep(1.2)
        rows = df.fillna("").astype(str).values.tolist()
        for i in range(0, len(rows), 40):
            ws.append_rows(rows[i:i+40], value_input_option="USER_ENTERED")
            if i + 40 < len(rows): time.sleep(1.2)
        print(f"  ✅ {sheet_name} → {len(df)} wierszy")
        return True
    except Exception as e:
        print(f"  ❌ BŁĄD {sheet_name}: {e}")
        return False

TEAM_NAME_OVERRIDES = {
    "Terville Florange W": "Terville-Florange OC",
    "Terville Florange":   "Terville-Florange OC",
    "Terville-Florange":   "Terville-Florange OC",
    "Béziers W":           "Beziers Volley",
    "Béziers":             "Beziers Volley",
    "Beziers W":           "Beziers Volley",
    "Beziers Volley":      "Beziers Volley",
}

def get_canonical_team_name(raw):
    if not raw: return "UNKNOWN"
    stripped = raw.strip()
    if stripped in TEAM_NAME_OVERRIDES:
        return TEAM_NAME_OVERRIDES[stripped]
    norm = normalize_name(stripped)
    for k, val in TEAM_NAME_OVERRIDES.items():
        if normalize_name(k) == norm: return val
    return normalize_name(stripped)

# ── Połącz z Sheets jeśli BOOK nie jest zdefiniowany ─────────
import os, json as _json
try:
    BOOK
    print("✅ BOOK już połączony")
except NameError:
    import gspread
    from google.oauth2.service_account import Credentials
    SA_PATH = "/content/service_account.json"
    SCOPES  = ["https://www.googleapis.com/auth/spreadsheets",
               "https://www.googleapis.com/auth/drive"]
    creds   = Credentials.from_service_account_file(SA_PATH, scopes=SCOPES)
    client  = gspread.authorize(creds)
    BOOK    = client.open_by_key("1x80zxUeYfKk9_kvsgeR1D9ayoKfaTPE90wNOGvuodnI")
    print(f"✅ Połączono z: {BOOK.title}")

# ── Wczytaj players_master i expected_lineups ze Sheets ───────
try:
    players_master_df
    print(f"✅ players_master: {len(players_master_df)} wierszy")
except NameError:
    import time
    def ws_to_df(ws):
        data = ws.get_all_values()
        if len(data) < 2: return pd.DataFrame()
        return pd.DataFrame(data[1:], columns=data[0])
    print("📥 Wczytuję players_master ze Sheets...")
    players_master_df = ws_to_df(BOOK.worksheet("players_master"))
    print(f"   players_master: {len(players_master_df)} wierszy")

try:
    expected_lineups_df
    print(f"✅ expected_lineups: {len(expected_lineups_df)} wierszy")
except NameError:
    import time
    def ws_to_df(ws):
        data = ws.get_all_values()
        if len(data) < 2: return pd.DataFrame()
        return pd.DataFrame(data[1:], columns=data[0])
    print("📥 Wczytuję expected_lineups ze Sheets...")
    expected_lineups_df = ws_to_df(BOOK.worksheet("expected_lineups"))
    print(f"   expected_lineups: {len(expected_lineups_df)} wierszy")

# ═══════════════════════════════════════════════════════════════
# LIVE LINEUP — mecz 9086: Terville Florange vs Béziers
# Źródło: protokół meczowy LNV, 14 marca 2026, set 1
# ═══════════════════════════════════════════════════════════════

MATCH_ID   = "9086"
MATCH_DATE = "2026-03-14"
SOURCE     = "manual_protocol"

TERVILLE_ID   = make_team_id(get_canonical_team_name("Terville Florange W"))
BEZIERS_ID    = make_team_id(get_canonical_team_name("Béziers W"))
TERVILLE_NAME = "Terville-Florange OC"
BEZIERS_NAME  = "Beziers Volley"

print(f"Terville team_id: {TERVILLE_ID}")
print(f"Beziers  team_id: {BEZIERS_ID}")

# ── Składy startowe set 1 ─────────────────────────────────────
# Format: (slot, nr_koszulki, raw_name, position)
TERVILLE_LINEUP = [
    ("S",    "14", "Kulikova Tatiana",       "S"),
    ("OPP",  "24", "Segovia Dayana",         "OPP"),
    ("OH1",  "9",  "Chillingworth Grace",    "OH"),
    ("OH2",  "99", "Wilson Megan",           "OH"),
    ("MB1",  "11", "Hendrickson Camille",    "MB"),
    ("MB2",  "1",  "Strand Isabel",          "MB"),
    ("L",    "17", "Coulet Sarah",           "L"),
]

BEZIERS_LINEUP = [
    ("S",    "2",  "Pierret Meline",                  "S"),
    ("OPP",  "21", "Elouga Eva-Brooklyn",             "OPP"),
    ("OH1",  "17", "Debell Ciara",                    "OH"),
    ("OH2",  "6",  "Carabali De La Cruz Valerin",     "OH"),
    ("MB1",  "7",  "Ahmed Houssameldien Nada",        "MB"),
    ("MB2",  "14", "Mercado Erika",                   "MB"),
    ("L",    "23", "Romati Sara",                     "L"),
]

def build_live_rows(match_id, team_id, team_name, lineup, players_master_df, expected_lineups_df):
    rows = []
    now  = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

    pid_lookup = {}
    if not players_master_df.empty:
        for _, r in players_master_df.iterrows():
            pid_lookup[r["normalized_player_name"].lower()] = r["player_id"]

    typical_ids = set()
    if not expected_lineups_df.empty:
        typical_ids = set(
            expected_lineups_df[expected_lineups_df["team_id"] == team_id]["player_id"].tolist()
        )

    for slot, player_no, raw_name, position in lineup:
        norm_name = normalize_name(raw_name)

        resolved_pid  = pid_lookup.get(norm_name.lower())
        resolved_conf = "high"
        if not resolved_pid:
            from rapidfuzz import process as rfp, fuzz as rff
            candidates = list(pid_lookup.keys())
            if candidates:
                match = rfp.extractOne(
                    norm_name.lower(), candidates,
                    scorer=rff.token_sort_ratio, score_cutoff=80
                )
                if match:
                    resolved_pid  = pid_lookup[match[0]]
                    resolved_conf = "medium"
        if not resolved_pid:
            resolved_pid  = make_player_id(norm_name, team_id)
            resolved_conf = "low"

        is_missing = str(resolved_pid not in typical_ids) if typical_ids else "False"

        rows.append({
            "match_id":                    match_id,
            "team_id":                     team_id,
            "team_name":                   team_name,
            "source_type":                 SOURCE,
            "source_timestamp":            now,
            "set_no":                      "1",
            "slot_name":                   slot,
            "player_no":                   player_no,
            "live_player_name_raw":        raw_name,
            "live_player_name_normalized": norm_name,
            "resolved_player_id":          resolved_pid,
            "resolved_position":           position,
            "resolution_confidence":       resolved_conf,
            "is_missing_from_typical":     is_missing,
            "notes":                       "match_9086_set1",
        })
    return rows

rows_terville = build_live_rows(MATCH_ID, TERVILLE_ID, TERVILLE_NAME, TERVILLE_LINEUP, players_master_df, expected_lineups_df)
rows_beziers  = build_live_rows(MATCH_ID, BEZIERS_ID,  BEZIERS_NAME,  BEZIERS_LINEUP,  players_master_df, expected_lineups_df)

all_rows = rows_terville + rows_beziers
live_lineups_df = pd.DataFrame(all_rows)

print(f"\n✅ Live lineup mecz {MATCH_ID}: {len(live_lineups_df)} wierszy")
print(live_lineups_df[[
    "slot_name","player_no","live_player_name_normalized","team_name",
    "resolved_position","resolution_confidence","is_missing_from_typical"
]].to_string(index=False))

print("\n💾 Zapisuję live_lineups do Sheets...")
save_df(BOOK, "live_lineups", live_lineups_df)
print("✅ Gotowe — live_lineups zaktualizowane z kolumną player_no")


## 🅶 FAZA G — Lineup Shock Engine

In [ ]:
def classify_shock_level(delta_pct: float) -> str:
    if delta_pct >= SHOCK_THRESHOLDS['neutral_or_positive']:
        return 'neutral_or_positive'
    elif delta_pct >= SHOCK_THRESHOLDS['low']:
        return 'low'
    elif delta_pct >= SHOCK_THRESHOLDS['medium']:
        return 'medium'
    else:
        return 'high'

def compute_lineup_shock(
    match_id:        str,
    team_id:         str,
    live_lineups:    pd.DataFrame,
    expected_lineups: pd.DataFrame,
    pis_rolling:     pd.DataFrame,
) -> Dict:
    """
    Faza G: Główny wzór lineup shock.
    LineupShock = (TIS_live - TIS_typical) / TIS_typical
    """
    # TIS_typical
    typical = expected_lineups[expected_lineups['team_id'] == team_id].copy()
    tis_typical = typical['slot_value'].astype(float).sum()
    typical_player_ids = set(typical['player_id'].tolist())
    typical_players_summary = typical['player_name'].tolist()

    # Live lineup dla tego meczu + drużyny
    live = live_lineups[
        (live_lineups['match_id'] == match_id) &
        (live_lineups['team_id'] == team_id)
    ].copy()

    if live.empty:
        return {'error': f'Brak live lineup dla match_id={match_id}, team_id={team_id}'}

    # Oblicz TIS_live: slot_value dla każdego live playera
    pis_lookup = pis_rolling.set_index('player_id')['pis_rolling'].to_dict()
    pos_lookup  = pis_rolling.set_index('player_id')['position'].to_dict()

    live_player_ids = set()
    tis_live = 0.0
    live_players_summary = []

    for _, row in live.iterrows():
        pid = row['resolved_player_id']
        live_player_ids.add(pid)
        live_players_summary.append(row['live_player_name_normalized'])

        pis_r = pis_lookup.get(pid, 0.0)
        pos   = pos_lookup.get(pid, row.get('resolved_position', 'UNKNOWN'))
        rw    = ROLE_WEIGHTS.get(pos, ROLE_WEIGHTS['UNKNOWN'])
        slot_val = pis_r * rw
        tis_live += slot_val

    # Missing / extra
    missing_ids   = typical_player_ids - live_player_ids
    extra_ids     = live_player_ids - typical_player_ids

    name_lookup = players_master_df.set_index('player_id')['normalized_player_name'].to_dict()
    missing_names = [name_lookup.get(pid, pid) for pid in missing_ids]
    extra_names   = [name_lookup.get(pid, pid) for pid in extra_ids]

    # Missing key roles
    missing_key_roles = []
    for pid in missing_ids:
        pos = pos_lookup.get(pid, 'UNKNOWN')
        if pos in KEY_ROLES:
            missing_key_roles.append(f'{name_lookup.get(pid, pid)}({pos})')

    delta_abs = round(tis_live - tis_typical, 4)
    delta_pct = round(delta_abs / tis_typical, 4) if tis_typical > 0 else 0.0
    lineup_shock = delta_pct
    shock_level  = classify_shock_level(delta_pct)

    # Comparison confidence
    if live['resolution_confidence'].eq('high').all():
        cmp_conf = 'high'
    elif live['resolution_confidence'].eq('low').any():
        cmp_conf = 'low'
    else:
        cmp_conf = 'medium'

    team_name = typical['team_name'].iloc[0] if not typical.empty else ''

    return {
        'match_id':               match_id,
        'team_id':                team_id,
        'team_name':              team_name,
        'typical_lineup_players': ', '.join(typical_players_summary),
        'live_lineup_players':    ', '.join(live_players_summary),
        'missing_players':        ', '.join(missing_names),
        'extra_players':          ', '.join(extra_names),
        'missing_key_roles':      ', '.join(missing_key_roles),
        'TIS_typical':            round(tis_typical, 4),
        'TIS_live':               round(tis_live, 4),
        'delta_TIS_abs':          delta_abs,
        'delta_TIS_pct':          delta_pct,
        'lineup_shock':           lineup_shock,
        'shock_level':            shock_level,
        'comparison_confidence':  cmp_conf,
        'created_at':             datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
    }

def run_shock_for_all_matches(
    live_lineups:    pd.DataFrame,
    expected_lineups: pd.DataFrame,
    pis_rolling:     pd.DataFrame,
) -> pd.DataFrame:
    results = []
    for (match_id, team_id) in live_lineups[['match_id','team_id']].drop_duplicates().values:
        res = compute_lineup_shock(match_id, team_id, live_lineups, expected_lineups, pis_rolling)
        if 'error' not in res:
            results.append(res)
    return pd.DataFrame(results)

shock_results_df = run_shock_for_all_matches(live_lineups_df, expected_lineups_df, pis_rolling_df)

print(f'\n✅ Faza G zakończona: {len(shock_results_df)} porównań lineup')
if not shock_results_df.empty:
    print(shock_results_df[[
        'match_id','team_name','TIS_typical','TIS_live',
        'delta_TIS_pct','lineup_shock','shock_level','missing_players'
    ]].to_string(index=False))

## 🅷 FAZA H — Pre-Match Fair Odds Adjustment

In [ ]:
# ═══════════════════════════════════════════════════════════════
# FAZA H — TEAM PRICING → team_pricing
# Główny arkusz decyzyjny.
# Odpowiada na pytanie: czy zmiana składu tworzy value vs rynek?
#
# Schemat kolumn zgodny z Google Sheets:
#   match_id, home_team_id, away_team_id,
#   tis_typical_home, tis_typical_away,
#   tis_live_home, tis_live_away,
#   lineup_shock_home, lineup_shock_away,
#   dtr_pre_home, dtr_pre_away,
#   lineup_adj_home, lineup_adj_away,
#   wp_home_base, wp_home_live,
#   true_odds_home, market_home_odds, market_prob_home,
#   edge_home_pp, ev_home, decision_home,
#   true_odds_away, edge_away_pp, decision_away
# ═══════════════════════════════════════════════════════════════

import math

# ── Stałe (synchronizacja z params) ──────────────────────────
HOME_ADV_DTR       = 3.0    # params!B9  — przewaga domowa
LOGISTIC_SCALE     = 8.0    # dzielnik w sigmoid (kalibrowalny)
LINEUP_ADJ_MULT    = 15.0   # shock → DTR adjustment (params!B9 multiplier)

def build_team_pricing(
    matches_df:      pd.DataFrame,
    expected_lineups: pd.DataFrame,
    live_lineups:    pd.DataFrame,
    shock_results:   pd.DataFrame,
    market_odds_dict: dict = None,
) -> pd.DataFrame:
    """
    Faza H: Buduje team_pricing — jeden wiersz na mecz.

    market_odds_dict: {match_id: {"home": float, "away": float}}
    Jeśli brak danych rynkowych — market_odds = None, edge = None.

    DTR placeholder v0:
      dtr_pre = TIS_typical (brak zewnętrznego źródła w PoC)
    """
    if market_odds_dict is None:
        market_odds_dict = {}

    # ── TIS typical per team ──────────────────────────────────
    tis_typical = {}
    if not expected_lineups.empty:
        for team_id, grp in expected_lineups.groupby("team_id"):
            tis_typical[team_id] = float(grp["slot_value"].astype(float).sum())

    # ── TIS live per (match_id, team_id) ─────────────────────
    tis_live = {}
    if not live_lineups.empty and "tis_contribution" in live_lineups.columns:
        for (mid, tid), grp in live_lineups.groupby(["match_id","team_id"]):
            tis_live[(mid, tid)] = float(grp["tis_contribution"].astype(float).sum())

    # ── Shock results lookup ──────────────────────────────────
    shock_lookup = {}
    if not shock_results.empty:
        for _, r in shock_results.iterrows():
            shock_lookup[(r["match_id"], r["team_id"])] = r

    # ── Pary meczowe ─────────────────────────────────────────
    m_cols = list(matches_df.columns)
    c_mid  = next((c for c in m_cols if "match_id"  in c.lower()), None)
    c_home = next((c for c in m_cols if "home_team" in c.lower()), None)
    c_away = next((c for c in m_cols if "away_team" in c.lower()), None)

    if not c_mid or not c_home or not c_away:
        log.warning("Brak kolumn match_id/home_team/away_team w matches_df")
        return pd.DataFrame()

    records = []

    for _, match in matches_df.iterrows():
        mid       = str(match[c_mid]).strip()
        home_raw  = str(match[c_home]).strip()
        away_raw  = str(match[c_away]).strip()

        if not mid or not home_raw or not away_raw:
            continue
        if home_raw in ("", "?") or away_raw in ("", "?"):
            continue

        home_id = make_team_id(get_canonical_team_name(home_raw))
        away_id = make_team_id(get_canonical_team_name(away_raw))

        # TIS typical
        tis_typ_h = tis_typical.get(home_id, 0.0)
        tis_typ_a = tis_typical.get(away_id, 0.0)

        # TIS live (fallback = typical jeśli brak live)
        tis_liv_h = tis_live.get((mid, home_id), tis_typ_h)
        tis_liv_a = tis_live.get((mid, away_id), tis_typ_a)

        # Lineup shock
        shock_h = (tis_liv_h - tis_typ_h) / tis_typ_h if tis_typ_h > 0 else 0.0
        shock_a = (tis_liv_a - tis_typ_a) / tis_typ_a if tis_typ_a > 0 else 0.0

        # DTR pre v0 = TIS_typical (placeholder)
        dtr_pre_h = round(tis_typ_h, 4)
        dtr_pre_a = round(tis_typ_a, 4)

        # Lineup adjustment = shock * 15
        lineup_adj_h = round(shock_h * LINEUP_ADJ_MULT, 4)
        lineup_adj_a = round(shock_a * LINEUP_ADJ_MULT, 4)

        # Base win probability (bez lineup adj)
        # wp_home_base = sigmoid((dtr_pre_h + HOME_ADV - dtr_pre_a) / scale)
        wp_base = sigmoid(
            ((dtr_pre_h + HOME_ADV_DTR) - dtr_pre_a) / LOGISTIC_SCALE
        )

        # Live win probability (z lineup adj)
        wp_live = clamp(sigmoid(
            (((dtr_pre_h + lineup_adj_h) + HOME_ADV_DTR)
             - (dtr_pre_a + lineup_adj_a)) / LOGISTIC_SCALE
        ))

        # True odds
        true_odds_h = round(1.0 / wp_live, 3)        if wp_live > 0    else 999.0
        true_odds_a = round(1.0 / (1.0 - wp_live), 3) if wp_live < 1.0 else 999.0

        # Market odds
        mkt = market_odds_dict.get(mid, {})
        mkt_odds_h = mkt.get("home", None)
        mkt_odds_a = mkt.get("away", None)

        mkt_prob_h   = round(1.0 / mkt_odds_h, 4)   if mkt_odds_h else None
        edge_h       = round(wp_live - mkt_prob_h, 4) if mkt_prob_h else None
        ev_h         = round(wp_live * mkt_odds_h - 1, 4) if mkt_odds_h else None
        edge_a_raw   = round((1.0 - wp_live) - (1.0 / mkt_odds_a), 4) if mkt_odds_a else None

        decision_h = decision_label(edge_h) if edge_h is not None else "NO ODDS"
        decision_a = decision_label(edge_a_raw) if edge_a_raw is not None else "NO ODDS"

        records.append({
            "match_id":           mid,
            "home_team_id":       home_id,
            "away_team_id":       away_id,
            "tis_typical_home":   round(tis_typ_h, 4),
            "tis_typical_away":   round(tis_typ_a, 4),
            "tis_live_home":      round(tis_liv_h, 4),
            "tis_live_away":      round(tis_liv_a, 4),
            "lineup_shock_home":  round(shock_h, 4),
            "lineup_shock_away":  round(shock_a, 4),
            "dtr_pre_home":       dtr_pre_h,
            "dtr_pre_away":       dtr_pre_a,
            "lineup_adj_home":    lineup_adj_h,
            "lineup_adj_away":    lineup_adj_a,
            "wp_home_base":       round(wp_base, 4),
            "wp_home_live":       round(wp_live, 4),
            "true_odds_home":     true_odds_h,
            "market_home_odds":   mkt_odds_h if mkt_odds_h else "",
            "market_prob_home":   mkt_prob_h if mkt_prob_h else "",
            "edge_home_pp":       edge_h if edge_h is not None else "",
            "ev_home":            ev_h if ev_h is not None else "",
            "decision_home":      decision_h,
            "true_odds_away":     true_odds_a,
            "edge_away_pp":       edge_a_raw if edge_a_raw is not None else "",
            "decision_away":      decision_a,
        })

    df = pd.DataFrame(records)
    log.info(f"team_pricing: {len(df)} meczów")
    return df

# ── Uruchom ────────────────────────────────────────────────────
# market_odds_dict — wstaw prawdziwe kursy bukmacherskie gdy dostępne
# Format: {"match_id": {"home": 1.85, "away": 2.10}}
MARKET_ODDS = {}   # ← placeholder v0, uzupełnij przed produkcją

team_pricing_df = build_team_pricing(
    matches_df       = matches_df,
    expected_lineups  = expected_lineups_df,
    live_lineups     = live_lineups_df,
    shock_results    = shock_results_df,
    market_odds_dict = MARKET_ODDS,
)

print(f"\n✅ Faza H zakończona: {len(team_pricing_df)} meczów w team_pricing")
if not team_pricing_df.empty:
    print("\n📋 Podgląd kolumn decyzyjnych:")
    show_cols = [
        "match_id","tis_typical_home","tis_typical_away",
        "lineup_shock_home","lineup_shock_away",
        "wp_home_base","wp_home_live",
        "true_odds_home","true_odds_away","decision_home","decision_away"
    ]
    available = [c for c in show_cols if c in team_pricing_df.columns]
    print(team_pricing_df[available].head(10).to_string(index=False))

## 🅸 FAZA I — Alert Log → `alerts_log`

In [ ]:
import uuid

ALERTS_HEADERS = [
    'alert_id', 'timestamp', 'match_id',
    'team_side_or_bet_side', 'home_team', 'away_team',
    'typical_lineup_summary', 'live_lineup_summary',
    'missing_players', 'delta_tis', 'lineup_shock',
    'model_prob', 'market_prob', 'true_odds', 'market_odds',
    'edge', 'confidence', 'decision', 'rationale', 'status'
]

def build_rationale(
    shock_result: Dict,
    odds_result:  Dict,
) -> str:
    """Generuje czytelny, audytowalny rationale dla alertu."""
    parts = []

    shock_level  = shock_result.get('shock_level', '?')
    delta_pct    = shock_result.get('delta_TIS_pct', 0.0)
    missing      = shock_result.get('missing_players', '')
    missing_key  = shock_result.get('missing_key_roles', '')
    team         = shock_result.get('team_name', '?')

    parts.append(f"Lineup shock {shock_level.upper()} dla {team}.")
    parts.append(f"TIS_live vs TIS_typical: delta={delta_pct*100:.1f}%.")

    if missing:
        parts.append(f"Nieobecne zawodniczki: {missing}.")
    if missing_key:
        parts.append(f"⚠️ Brak kluczowych ról: {missing_key}.")

    edge = odds_result.get('edge')
    if edge is not None:
        parts.append(f"Edge vs rynek: {edge*100:.1f}pp ({odds_result.get('edge_band','?')}).")
    else:
        parts.append('Brak danych rynkowych (placeholder base_prob=0.50).')

    return ' '.join(parts)

def build_decision(shock_level: str, edge_band: str) -> str:
    if edge_band == 'strong_alert':
        return 'STRONG_ALERT'
    elif edge_band == 'value':
        return 'VALUE_BET'
    elif edge_band == 'watch':
        return 'WATCH'
    elif shock_level in ('high', 'medium'):
        return 'MONITOR_NO_ODDS'
    else:
        return 'NO_BET'

def generate_alerts(
    shock_results:    pd.DataFrame,
    matches_df:       pd.DataFrame,
    base_prob:        float = 0.50,
    market_odds_dict: Optional[Dict[str, float]] = None,
) -> pd.DataFrame:
    """
    Faza I: Generuje alerts_log.
    market_odds_dict: {match_id: decimal_odds} — opcjonalne
    """
    if shock_results.empty:
        return pd.DataFrame(columns=ALERTS_HEADERS)

    if market_odds_dict is None:
        market_odds_dict = {}

    # Lookup home/away z matches_df
    match_info: Dict[str, Dict] = {}
    if not matches_df.empty:
        m_cols = list(matches_df.columns)
        c_mid  = next((c for c in m_cols if 'match_id' in c.lower()), None)
        c_home = next((c for c in m_cols if 'home_team' in c.lower()), None)
        c_away = next((c for c in m_cols if 'away_team' in c.lower()), None)
        if c_mid and c_home and c_away:
            for _, r in matches_df.iterrows():
                match_info[str(r[c_mid])] = {
                    'home': str(r[c_home]), 'away': str(r[c_away])
                }

    records = []
    now_str = datetime.now().strftime('%Y-%m-%d %H:%M:%S')

    for _, sr in shock_results.iterrows():
        shock_dict  = sr.to_dict()
        match_id    = shock_dict.get('match_id', '')
        mkt_odds    = market_odds_dict.get(match_id)
        odds_result = compute_fair_odds(shock_dict, base_prob, mkt_odds)

        info = match_info.get(match_id, {'home': '?', 'away': '?'})

        decision  = build_decision(shock_dict.get('shock_level',''), odds_result.get('edge_band',''))
        rationale = build_rationale(shock_dict, odds_result)
        conf      = shock_dict.get('comparison_confidence', 'low')

        records.append({
            'alert_id':              str(uuid.uuid4())[:16],
            'timestamp':             now_str,
            'match_id':              match_id,
            'team_side_or_bet_side': shock_dict.get('team_name',''),
            'home_team':             info['home'],
            'away_team':             info['away'],
            'typical_lineup_summary': shock_dict.get('typical_lineup_players',''),
            'live_lineup_summary':    shock_dict.get('live_lineup_players',''),
            'missing_players':        shock_dict.get('missing_players',''),
            'delta_tis':              shock_dict.get('delta_TIS_pct',''),
            'lineup_shock':           shock_dict.get('lineup_shock',''),
            'model_prob':             odds_result.get('model_prob',''),
            'market_prob':            odds_result.get('market_prob',''),
            'true_odds':              odds_result.get('true_odds',''),
            'market_odds':            odds_result.get('market_odds',''),
            'edge':                   odds_result.get('edge',''),
            'confidence':             conf,
            'decision':               decision,
            'rationale':              rationale,
            'status':                 'new',
        })

    df = pd.DataFrame(records)
    log.info(f'alerts_log: {len(df)} alertów wygenerowanych')
    return df

# ── Uruchom z mock market odds dla demonstracji ────────────────
# W v1: podpnij tu prawdziwe kursy bukmacherskie
MOCK_MARKET_ODDS = {}  # np. {'MOCK_9001': 2.10}

alerts_df = generate_alerts(
    shock_results_df, matches_df,
    base_prob=0.50,
    market_odds_dict=MOCK_MARKET_ODDS
)

print(f'\n✅ Faza I zakończona: {len(alerts_df)} alertów')
if not alerts_df.empty:
    print('\n📋 WYGENEROWANE ALERTY:')
    print('='*80)
    for _, row in alerts_df.iterrows():
        print(f"Alert:    {row['alert_id']}")
        print(f"Mecz:     {row['match_id']} | {row['home_team']} vs {row['away_team']}")
        print(f"Drużyna:  {row['team_side_or_bet_side']}")
        print(f"Shock:    {row['lineup_shock']} ({row['delta_tis']})")      
        print(f"Model:    prob={row['model_prob']} | odds={row['true_odds']}")
        print(f"Decyzja:  {row['decision']}")
        print(f"Rationale: {row['rationale']}")
        print(f"Braki:    {row['missing_players'] or '(brak)'}")  
        print('-'*80)

## 💾 CELL KOŃCOWY — Zapis wszystkich wyników do Google Sheets

## ⚙️ PARAMS — Zakładka sterująca (wagi i progi)

## 📊 ANALIZA MECZU 9086 — Terville-Florange W vs Beziers W

# ── Zregeneruj alerts_log dla meczu 9086 ─────────────────────
print("\n💾 Generuję alert dla meczu 9086...")
import uuid, time

now_str = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

# Rationale
rationale_parts = []
rationale_parts.append(f"Mecz: {HOME_TEAM_NAME} vs {AWAY_TEAM_NAME}.")
rationale_parts.append(f"WP home (model): {wp_live*100:.1f}% | True odds: {true_odds_h}.")

if tis_liv_a and shock_a != 0:
    rationale_parts.append(
        f"Lineup shock {AWAY_TEAM_NAME}: {shock_a*100:.1f}% ({shock_level(shock_a)})."
    )
if tis_liv_h and shock_h != 0:
    rationale_parts.append(
        f"Lineup shock {HOME_TEAM_NAME}: {shock_h*100:.1f}% ({shock_level(shock_h)})."
    )
if market_home_odds and edge_h is not None:
    rationale_parts.append(
        f"Kurs rynkowy (home): {market_home_odds} "
        f"| Market prob: {1/market_home_odds*100:.1f}% "
        f"| Edge: {edge_h*100:+.1f}pp."
    )
    if edge_h > 0.04:
        rationale_parts.append("Edge powyżej progu VALUE — rynek niedowartościował korektę składu.")
else:
    rationale_parts.append("Brak kursu rynkowego — edge nieobliczony.")

alert = {
    "alert_id":                str(uuid.uuid4())[:16],
    "timestamp":               now_str,
    "match_id":                MATCH_ID,
    "team_side_or_bet_side":   HOME_TEAM_NAME,
    "home_team":               HOME_TEAM_NAME,
    "away_team":               AWAY_TEAM_NAME,
    "typical_lineup_summary":  ", ".join([p[1] for p in (typ_h_players or [])]),
    "live_lineup_summary":     ", ".join([p[1] for p in (liv_h_players or [])]),
    "missing_players":         "",
    "delta_tis":               str(round(shock_h, 4)),
    "lineup_shock":            str(round(shock_h, 4)),
    "model_prob":              str(round(wp_live, 4)),
    "market_prob":             str(round(1/market_home_odds, 4)) if market_home_odds else "",
    "true_odds":               str(true_odds_h),
    "market_odds":             str(market_home_odds) if market_home_odds else "",
    "edge":                    str(round(edge_h, 4)) if edge_h is not None else "",
    "confidence":              "medium",
    "decision":                decision,
    "rationale":               " ".join(rationale_parts),
    "status":                  "new",
}

# Also alert for away team
shock_away_level = shock_level(shock_a)
alert_away = {
    "alert_id":                str(uuid.uuid4())[:16],
    "timestamp":               now_str,
    "match_id":                MATCH_ID,
    "team_side_or_bet_side":   AWAY_TEAM_NAME,
    "home_team":               HOME_TEAM_NAME,
    "away_team":               AWAY_TEAM_NAME,
    "typical_lineup_summary":  ", ".join([p[1] for p in (typ_a_players or [])]),
    "live_lineup_summary":     ", ".join([p[1] for p in (liv_a_players or [])]),
    "missing_players":         "",
    "delta_tis":               str(round(shock_a, 4)),
    "lineup_shock":            str(round(shock_a, 4)),
    "model_prob":              str(round(1 - wp_live, 4)),
    "market_prob":             "",
    "true_odds":               str(true_odds_a),
    "market_odds":             "",
    "edge":                    "",
    "confidence":              "medium",
    "decision":                "MONITOR" if shock_away_level in ("medium","high") else "NO BET",
    "rationale":               (
        f"Lineup shock {AWAY_TEAM_NAME}: {shock_a*100:.1f}% ({shock_away_level}). "
        f"TIS_live={eff_liv_a:.2f} vs TIS_typical={tis_typ_a:.2f}. "
        f"Model WP away: {(1-wp_live)*100:.1f}% | True odds away: {true_odds_a}."
    ),
    "status":                  "new",
}

# Zapisz do alerts_log — dopisz (nie nadpisuj)
ws_alerts = BOOK.worksheet("alerts_log")
all_alerts = ws_alerts.get_all_values()
headers_a  = [h.strip() for h in all_alerts[0]] if all_alerts else []

def alert_to_row(a, headers):
    return [str(a.get(h, "")) for h in headers]

if headers_a:
    ws_alerts.append_row(alert_to_row(alert, headers_a),      value_input_option="USER_ENTERED")
    time.sleep(1)
    ws_alerts.append_row(alert_to_row(alert_away, headers_a), value_input_option="USER_ENTERED")
    print(f"✅ 2 alerty dodane do alerts_log")
    print(f"   HOME: decision={alert['decision']} | edge={alert['edge']}")
    print(f"   AWAY: decision={alert_away['decision']} | shock={shock_a*100:.1f}%")
else:
    print("⚠️  alerts_log ma puste nagłówki — sprawdź zakładkę")


In [ ]:
# ═══════════════════════════════════════════════════════════════
# CLEANUP ALERTS — usuń stare alerty z mock/placeholder danych
# Uruchom raz przed ANALIZĄ MECZU 9086
# ═══════════════════════════════════════════════════════════════
import time

try: BOOK
except NameError:
    import gspread
    from google.oauth2.service_account import Credentials
    creds = Credentials.from_service_account_file("/content/service_account.json",
        scopes=["https://www.googleapis.com/auth/spreadsheets",
                "https://www.googleapis.com/auth/drive"])
    BOOK = gspread.authorize(creds).open_by_key("1x80zxUeYfKk9_kvsgeR1D9ayoKfaTPE90wNOGvuodnI")

ws_a  = BOOK.worksheet("alerts_log")
data  = ws_a.get_all_values()
hdrs  = data[0]
rows  = data[1:]

def ci(col):
    try: return hdrs.index(col)
    except: return None

i_rat  = ci("rationale") or ci("notes") or len(hdrs)-1
i_dec  = ci("decision")  or len(hdrs)-3
i_ts   = ci("timestamp") or 0

# Zachowaj tylko alerty z prawdziwymi danymi (nie placeholder/mock)
BAD_PATTERNS = [
    "placeholder base_prob",
    "base_prob=0.50",
    "Lineup shock NEUTRAL_OR_POSITIVE",
    "TIS_live vs TIS_typical: delta=0.0%",
    "Brak danych rynkowych",
    "MOCK",
    "mock",
]

kept   = []
removed = 0
for row in rows:
    rat = str(row[i_rat]).strip() if len(row) > i_rat else ""
    is_bad = any(p in rat for p in BAD_PATTERNS)
    if is_bad:
        removed += 1
    else:
        kept.append(row)

print(f"Alerty: {len(rows)} total → usuwam {removed} stałych/mock → zostaje {len(kept)}")

# Nadpisz alerts_log czystymi danymi
ws_a.clear()
time.sleep(1.5)
ws_a.update("A1", [hdrs], value_input_option="USER_ENTERED")
time.sleep(1)
if kept:
    ws_a.append_rows(kept, value_input_option="USER_ENTERED")
    time.sleep(1)
print(f"✅ alerts_log wyczyszczony: {len(kept)} prawdziwych alertów pozostało")
print("\nTeraz uruchom komórkę ANALIZA MECZU 9086 →")


In [ ]:
# ═══════════════════════════════════════════════════════════════
# ANALIZA MECZU 9086 — Terville-Florange W vs Beziers W
# Kalkulacje: TIS_expected, TIS_live, shock, WP, edge, EV
# Wymaga: Fazy C, D, Live Lineup cell
# ═══════════════════════════════════════════════════════════════

import pandas as pd, numpy as np, math, unicodedata, hashlib, re, uuid, time
from datetime import datetime

try: BOOK
except NameError:
    import gspread
    from google.oauth2.service_account import Credentials
    creds = Credentials.from_service_account_file("/content/service_account.json",
        scopes=["https://www.googleapis.com/auth/spreadsheets",
                "https://www.googleapis.com/auth/drive"])
    BOOK = gspread.authorize(creds).open_by_key("1x80zxUeYfKk9_kvsgeR1D9ayoKfaTPE90wNOGvuodnI")

def ws_to_df(ws):
    d = ws.get_all_values()
    return pd.DataFrame(d[1:], columns=d[0]) if len(d) > 1 else pd.DataFrame()

def safe_float(v, d=0.0):
    try: return float(str(v).strip().replace(",",".")) or d
    except: return d

def sigmoid(x):
    try: return 1.0 / (1.0 + math.exp(-x))
    except: return 0.0 if x < 0 else 1.0

MATCH_ID       = "9086"
HOME_TEAM_KW   = ["Terville", "terville"]
AWAY_TEAM_KW   = ["Bezier", "bezier"]
HOME_ADV       = 3.0
LOGISTIC_SCALE = 25.0
ROLE_WEIGHTS   = {"S":1.25,"OPP":1.20,"OH":1.10,"MB":1.00,"L":0.90,"UNKNOWN":0.85}

print("📥 Wczytuję dane...")
expected_df  = ws_to_df(BOOK.worksheet("expected_lineups"))
live_df      = ws_to_df(BOOK.worksheet("live_lineups"))
pr_df        = ws_to_df(BOOK.worksheet("player_rolling"))
pricing_df   = ws_to_df(BOOK.worksheet("team_pricing"))
print(f"   expected_lineups: {len(expected_df)} wierszy")
print(f"   live_lineups:     {len(live_df)} wierszy")
print(f"   player_rolling:   {len(pr_df)} wierszy")
print(f"   team_pricing:     {len(pricing_df)} wierszy")

# ── Znajdź wiersz meczu 9086 w team_pricing ──────────────────
match_row = pricing_df[pricing_df["match_id"].astype(str) == MATCH_ID]
market_home_odds = None

if not match_row.empty:
    # Priorytet: market_home_odds (kolumna Q) → inne kolumny z "odds/market/kurs"
    for col in ["market_home_odds","market_odds","mkt_odds","kurs_home","odds_home"]:
        if col in match_row.columns:
            f = safe_float(match_row[col].iloc[0])
            if 1.01 <= f <= 20.0:
                market_home_odds = f
                print(f"   ✅ market_home_odds z '{col}': {market_home_odds}")
                break

    # Fallback: szukaj w każdej kolumnie
    if market_home_odds is None:
        for col in pricing_df.columns:
            val = match_row[col].iloc[0]
            f = safe_float(val)
            if 1.01 <= f <= 20.0 and any(k in col.lower() for k in ['market','odds','kurs','mkt']):
                market_home_odds = f
                print(f"   ✅ Kurs fallback z '{col}': {market_home_odds}")
                break

    if market_home_odds:
        print(f"   ✅ Kurs rynkowy: {market_home_odds}")
    else:
        print(f"   ⚠️  Brak kursu w team_pricing — wpisz ręcznie w kolumnę Q dla meczu {MATCH_ID}")
        print(f"   Dostępne kolumny: {list(pricing_df.columns)}")
else:
    print(f"⚠️  Mecz {MATCH_ID} nie znaleziony w team_pricing")

# ── PIS lookup z player_rolling ──────────────────────────────
pis_map = {}
pos_map = {}
if not pr_df.empty:
    for _, r in pr_df.iterrows():
        pid = str(r.get("player_id",""))
        pis_map[pid] = safe_float(r.get("pis_rolling",0))
        pos_map[pid] = str(r.get("position","UNKNOWN"))
        # Also index by name
        nm = str(r.get("player_name","")).lower().strip()
        if nm: pis_map[nm] = safe_float(r.get("pis_rolling",0))

# ── TIS z expected_lineups ────────────────────────────────────
def calc_tis_expected(team_kw):
    rows = expected_df[expected_df["team_name"].str.contains(
        team_kw[0], case=False, na=False)]
    if rows.empty:
        return 0.0, []
    total = rows["slot_value"].astype(float).sum()
    players = rows[["slot_name","player_name","pis_rolling","start_rate","slot_value"]].values.tolist()
    return round(total, 4), players

# ── TIS z live_lineups ────────────────────────────────────────
def calc_tis_live(team_kw):
    live = live_df[
        (live_df["match_id"].astype(str) == MATCH_ID) &
        (live_df["team_name"].str.contains(team_kw[0], case=False, na=False))
    ]
    if live.empty:
        return None, []
    total = 0.0
    players = []
    for _, r in live.iterrows():
        pid  = str(r.get("resolved_player_id",""))
        nm   = str(r.get("live_player_name_normalized","") or r.get("live_player_name_raw","")).lower()
        pis  = pis_map.get(pid, pis_map.get(nm, 0.0))
        pos  = pos_map.get(pid, str(r.get("resolved_position","UNKNOWN")))
        rw   = ROLE_WEIGHTS.get(pos, 0.85)
        contrib = round(pis * rw, 4)
        total += contrib
        players.append([r.get("slot_name",""), r.get("live_player_name_normalized",""),
                        r.get("player_no",""), pos, f"PIS={pis:.2f}", f"={contrib:.2f}"])
    return round(total, 4), players

# ── Obliczenia ────────────────────────────────────────────────
tis_exp_h, exp_h_pl = calc_tis_expected(HOME_TEAM_KW)
tis_exp_a, exp_a_pl = calc_tis_expected(AWAY_TEAM_KW)
tis_liv_h, liv_h_pl = calc_tis_live(HOME_TEAM_KW)
tis_liv_a, liv_a_pl = calc_tis_live(AWAY_TEAM_KW)

eff_h = tis_liv_h if tis_liv_h and tis_liv_h > 0 else tis_exp_h
eff_a = tis_liv_a if tis_liv_a and tis_liv_a > 0 else tis_exp_a

shock_h = round((eff_h - tis_exp_h) / tis_exp_h, 4) if tis_exp_h > 0 else 0.0
shock_a = round((eff_a - tis_exp_a) / tis_exp_a, 4) if tis_exp_a > 0 else 0.0

la_h = shock_h * LOGISTIC_SCALE * 0.6
la_a = shock_a * LOGISTIC_SCALE * 0.6

wp_base = sigmoid(((tis_exp_h + HOME_ADV) - tis_exp_a) / LOGISTIC_SCALE)
wp_live  = max(0.01, min(0.99,
    sigmoid((((tis_exp_h + la_h) + HOME_ADV) - (tis_exp_a + la_a)) / LOGISTIC_SCALE)))

true_h = round(1.0/wp_live, 3)
true_a = round(1.0/(1-wp_live), 3)

def shock_label(s):
    if s >= -0.05: return "NEUTRAL/POSITIVE"
    if s >= -0.08: return "LOW"
    if s >= -0.15: return "MEDIUM"
    return "HIGH"

print(f"""
{'='*65}
ANALIZA MECZU {MATCH_ID}: Terville-Florange W vs Beziers W
{'='*65}

📊 TIS EXPECTED (skład optymalny):
   Home (Terville):  {tis_exp_h:.2f}
   Away (Beziers):   {tis_exp_a:.2f}

🔴 TIS LIVE (potwierdzony skład):
   Home (Terville):  {tis_liv_h if tis_liv_h else 'brak danych'} {'← z live_lineups' if tis_liv_h else '← fallback na expected'}
   Away (Beziers):   {tis_liv_a if tis_liv_a else 'brak danych'}

⚡ LINEUP SHOCK:
   Home: {shock_h*100:+.1f}% ({shock_label(shock_h)})
   Away: {shock_a*100:+.1f}% ({shock_label(shock_a)})

🎯 MODEL PROBABILITY:
   WP home (base):  {wp_base*100:.1f}%  (bez korekty live)
   WP home (live):  {wp_live*100:.1f}%  (z korektą lineup shock)
   True odds home:  {true_h}
   True odds away:  {true_a}
""")

edge_h = ev_h = decision = None
if market_home_odds and market_home_odds > 1.0:
    mkt_prob = 1.0 / market_home_odds
    edge_h   = round(wp_live - mkt_prob, 4)
    ev_h     = round(wp_live * market_home_odds - 1, 4)
    if edge_h > 0.06 and shock_label(shock_a) in ("MEDIUM","HIGH"):
        decision = "STRONG BET"
    elif edge_h > 0.04: decision = "VALUE BET"
    elif edge_h > 0.02: decision = "WATCH"
    else: decision = "NO BET"

    print(f"""💰 EDGE vs RYNEK:
   Kurs rynkowy (home): {market_home_odds}
   Market implied prob: {mkt_prob*100:.1f}%
   Model prob (live):   {wp_live*100:.1f}%
   Edge:                {edge_h*100:+.1f}pp
   EV (1 jedn.):        {ev_h:+.3f}

   ★ REKOMENDACJA: {decision} ★
""")
else:
    decision = "NO ODDS"
    print("⚠️  Brak kursu rynkowego")

# ── Skład expected vs live ────────────────────────────────────
print(f"\n📋 EXPECTED LINEUP vs LIVE — Terville-Florange W")
print(f"{'Slot':5} {'Expected':25} {'Live':25} {'Diff TIS':8}")
print("-"*70)
exp_by_slot = {str(r[0]):r for r in exp_h_pl}
liv_by_slot = {str(r[0]):r for r in liv_h_pl}
for slot in ["S","OPP","OH1","OH2","MB1","MB2","L"]:
    ep = exp_by_slot.get(slot)
    lp = liv_by_slot.get(slot)
    ep_name = str(ep[1])[:24] if ep else "—"
    lp_name = str(lp[1])[:24] if lp else "—"
    ep_sv   = float(ep[4]) if ep else 0.0
    lp_sv   = float(str(lp[5]).replace("=","")) if lp else 0.0
    same    = "✅" if ep and lp and ep_name.lower() in lp_name.lower() else "↔" if lp else "✗"
    print(f"  {slot:4} {ep_name:25} {lp_name:25} {same}")

print(f"\n📋 EXPECTED LINEUP vs LIVE — Beziers W")
print(f"{'Slot':5} {'Expected':25} {'Live':25}")
print("-"*55)
exp_by_slot_a = {str(r[0]):r for r in exp_a_pl}
liv_by_slot_a = {str(r[0]):r for r in liv_a_pl}
for slot in ["S","OPP","OH1","OH2","MB1","MB2","L"]:
    ep = exp_by_slot_a.get(slot)
    lp = liv_by_slot_a.get(slot)
    ep_name = str(ep[1])[:24] if ep else "—"
    lp_name = str(lp[1])[:24] if lp else "—"
    same    = "✅" if ep and lp and ep_name.lower() in lp_name.lower() else "↔" if lp else "✗"
    print(f"  {slot:4} {ep_name:25} {lp_name:25} {same}")

# ── Zapisz do team_pricing ────────────────────────────────────
print("\n💾 Zapisuję do team_pricing...")
ws_p   = BOOK.worksheet("team_pricing")
vals   = ws_p.get_all_values()
hdrs   = [h.strip() for h in vals[0]]
def ci(n):
    try: return hdrs.index(n)
    except: return None

mid_ci = ci("match_id")
row_idx = None
for j, row in enumerate(vals[1:], start=2):
    if mid_ci is not None and str(row[mid_ci]).strip() == MATCH_ID:
        row_idx = j; break

updates = {
    "tis_typical_home": str(round(tis_exp_h,4)),
    "tis_typical_away": str(round(tis_exp_a,4)),
    "tis_live_home":    str(round(eff_h,4)),
    "tis_live_away":    str(round(eff_a,4)),
    "lineup_shock_home":str(round(shock_h,4)),
    "lineup_shock_away":str(round(shock_a,4)),
    "lineup_adj_home":  str(round(la_h,4)),
    "lineup_adj_away":  str(round(la_a,4)),
    "wp_home_base":     str(round(wp_base,4)),
    "wp_home_live":     str(round(wp_live,4)),
    "true_odds_home":   str(true_h),
    "true_odds_away":   str(true_a),
    "dtr_pre_home":     str(round(tis_exp_h,4)),
    "dtr_pre_away":     str(round(tis_exp_a,4)),
}
if edge_h is not None:
    updates.update({"edge_home_pp":str(edge_h), "ev_home":str(ev_h), "decision_home":decision})
if market_home_odds:
    updates["market_home_odds"] = str(market_home_odds)

if row_idx:
    for col_name, val in updates.items():
        c = ci(col_name)
        if c is not None:
            ws_p.update(chr(ord("A")+c)+str(row_idx), [[val]],
                        value_input_option="USER_ENTERED")
            time.sleep(0.4)
    print(f"✅ team_pricing zaktualizowany (wiersz {row_idx})")
    print(f"   WP_live={wp_live*100:.1f}% | true_odds={true_h} | edge={edge_h*100:+.1f}pp | {decision}")
else:
    print(f"⚠️  Mecz {MATCH_ID} nie znaleziony w team_pricing — dodaj wiersz ręcznie")

# ── Alert log ─────────────────────────────────────────────────
print("\n💾 Dopisuję alerty...")
ws_a  = BOOK.worksheet("alerts_log")
hdrs_a = [h.strip() for h in ws_a.get_all_values()[0]]

def alert_row(side, team, mp, todds, edge, ev, dec, rationale):
    a = {
        "alert_id": str(uuid.uuid4())[:16],
        "timestamp": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
        "match_id": MATCH_ID,
        "team_side_or_bet_side": team,
        "home_team": "Terville-Florange W",
        "away_team": "Beziers W",
        "model_prob": str(round(mp,4)),
        "market_prob": str(round(1/market_home_odds,4)) if market_home_odds else "",
        "true_odds": str(todds),
        "market_odds": str(market_home_odds) if market_home_odds else "",
        "edge": str(edge) if edge is not None else "",
        "ev_home": str(ev) if ev is not None else "",
        "confidence": "medium",
        "decision": dec or "",
        "rationale": rationale,
        "status": "new",
    }
    return [str(a.get(h,"")) for h in hdrs_a]

r_h = (f"TIS_exp={tis_exp_h:.2f} TIS_live={eff_h:.2f} shock_home={shock_h*100:+.1f}% "
       f"WP_live={wp_live*100:.1f}% true_odds={true_h}")
r_a = (f"TIS_exp={tis_exp_a:.2f} TIS_live={eff_a:.2f} shock_away={shock_a*100:+.1f}% "
       f"WP_away={(1-wp_live)*100:.1f}% true_odds={true_a}")
if market_home_odds and edge_h is not None:
    r_h += f" market={market_home_odds} edge={edge_h*100:+.1f}pp EV={ev_h:+.3f}"

ws_a.append_row(alert_row("home","Terville-Florange W",wp_live,true_h,edge_h,ev_h,decision,r_h),
                value_input_option="USER_ENTERED")
time.sleep(1)
ws_a.append_row(alert_row("away","Beziers W",1-wp_live,true_a,None,None,
    "MONITOR" if shock_label(shock_a) in ("MEDIUM","HIGH") else "NO BET", r_a),
                value_input_option="USER_ENTERED")
print("✅ 2 alerty dodane do alerts_log")
print("="*65)


In [ ]:
PARAMS = [
    # ── PIS weights ───────────────────────────────────────────
    ("pis_attack_w",        1.00),   # waga punktów z ataku
    ("pis_block_w",         1.30),   # waga bloków
    ("pis_ace_w",           1.20),   # waga asów serwisowych
    ("pis_error_w",         0.80),   # waga błędów (odjemnik)
    # ── Role weights ──────────────────────────────────────────
    ("role_w_S",            1.25),   # rozgrywająca
    ("role_w_OPP",          1.20),   # atakująca przyjmująca
    ("role_w_OH",           1.10),   # przyjmująca atakująca
    ("role_w_MB",           1.00),   # środkowa
    ("role_w_L",            0.90),   # libero
    ("role_w_UNKNOWN",      0.85),   # nieznana pozycja
    # ── Starter proxy weights ─────────────────────────────────
    ("starter_role_w",      1.00),   # starter
    ("rotation_role_w",     0.70),   # rotacyjna
    ("bench_role_w",        0.40),   # rezerwowa
    # ── Rolling PIS weights ───────────────────────────────────
    ("pis_w_season",        0.50),   # waga średniej sezonu
    ("pis_w_last5",         0.30),   # waga ostatnich 5 meczów
    ("pis_w_last3",         0.20),   # waga ostatnich 3 meczów
    # ── Lineup Shock thresholds ───────────────────────────────
    ("lineup_shock_light",  -0.05),  # powyżej → neutral/positive
    ("lineup_shock_medium", -0.08),  # powyżej → low shock
    ("lineup_shock_strong", -0.15),  # powyżej → medium shock
                                     # poniżej → high shock
    # ── Fair odds adjustment ──────────────────────────────────
    ("shock_sensitivity",   0.60),   # % delta_TIS przekazywany do prob
    ("max_shock_adj",       0.12),   # max korekta prawdopodobieństwa
    # ── Edge alert bands ──────────────────────────────────────
    ("edge_watch_pp",       0.02),   # min edge → WATCH
    ("edge_bet_pp",         0.04),   # min edge → VALUE_BET
    ("edge_strong_pp",      0.06),   # min edge → STRONG_ALERT
    # ── Home advantage ────────────────────────────────────────
    ("home_adv_dtr",        3.00),   # placeholder DTR korekta domowa
    # ── Confidence thresholds ─────────────────────────────────
    ("conf_low_matches",    2),      # <= N meczów → low confidence
    ("conf_medium_matches", 4),      # <= N meczów → medium confidence
    # ── Fuzzy matching ────────────────────────────────────────
    ("fuzzy_alias_threshold",  88),  # min score dla alias detection
    ("fuzzy_live_threshold",   82),  # min score dla live name resolve
]

def write_params_to_sheets(book) -> None:
    """Zapisuje PARAMS do zakładki params w Google Sheets."""
    ws = ensure_worksheet(book, "params", rows=60, cols=3)
    ws.clear()
    time.sleep(DELAY_WRITE)
    headers = [["parameter", "value"]]
    ws.update("A1:B1", headers, value_input_option="USER_ENTERED")
    time.sleep(DELAY_WRITE)
    rows = [[p, str(v)] for p, v in PARAMS]
    ws.update(f"A2:B{len(rows)+1}", rows, value_input_option="USER_ENTERED")
    log.info(f"  → params: {len(rows)} parametrów zapisanych")

write_params_to_sheets(BOOK)
print(f"✅ params: {len(PARAMS)} parametrów zapisanych do Google Sheets")
print(f"{'parameter':30s} {'value':>8}")
print("-" * 40)
for p, v in PARAMS:
    print(f"{p:30s} {str(v):>8}")

In [ ]:
# ═══════════════════════════════════════════════════════════════
# DIAGNOSTYKA — sprawdź rozmiary DataFrame przed zapisem
# ═══════════════════════════════════════════════════════════════
dfs_to_check = {
    "teams_master":          teams_master_df,
    "player_match_features": player_match_features_df,
    "player_rolling":        player_rolling_df,
    "team_pricing":          team_pricing_df,
    "players_master":        players_master_df,
    "expected_lineups":       expected_lineups_df,
    "team_baselines":        team_baselines_df,
    "live_lineups":          live_lineups_df,
    "alerts_log":            alerts_df,
}

print("📊 Rozmiary DataFrame przed zapisem:")
print(f"  {'Zakładka':<30} {'Wiersze':>8}  {'Kolumny':>8}  {'Status'}")
print("  " + "-"*60)
all_ok = True
for name, df in dfs_to_check.items():
    if df is None:
        status = "❌ None"
        all_ok = False
    elif df.empty:
        status = "⚠️  PUSTY"
        all_ok = False
    else:
        status = "✅ OK"
    rows = len(df) if df is not None else 0
    cols = len(df.columns) if df is not None and not df.empty else 0
    print(f"  {name:<30} {rows:>8}  {cols:>8}  {status}")

if all_ok:
    print("\n✅ Wszystkie DataFrames gotowe do zapisu!")
else:
    print("\n⚠️  Niektóre DataFrames są puste — sprawdź błędy w poprzednich komórkach.")

In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELL KOŃCOWY — Zapis wszystkich wyników do Google Sheets
# ═══════════════════════════════════════════════════════════════

print("💾 Zapis do Google Sheets...\n")

results = {}
results["teams_master"]          = save_df(BOOK, "teams_master",          teams_master_df)
time.sleep(1.5)
results["player_match_features"] = save_df(BOOK, "player_match_features", player_match_features_df)
time.sleep(1.5)
results["player_rolling"]        = save_df(BOOK, "player_rolling",        player_rolling_df)
time.sleep(1.5)
# team_pricing: zachowaj ręcznie wpisane kolumny użytkownika (market_home_odds itp.)
try:
    ws_tp   = BOOK.worksheet("team_pricing")
    tp_live = ws_tp.get_all_values()
    tp_hdrs = tp_live[0] if tp_live else []
    USER_COLS = ["market_home_odds","market_away_odds","market_odds","notes_manual"]
    # Zbuduj mapę match_id → {col: val} dla kolumn użytkownika
    mid_idx  = tp_hdrs.index("match_id") if "match_id" in tp_hdrs else None
    user_vals = {}
    if mid_idx is not None:
        for row in tp_live[1:]:
            mid = str(row[mid_idx]).strip() if len(row) > mid_idx else ""
            if not mid: continue
            user_vals[mid] = {}
            for uc in USER_COLS:
                if uc in tp_hdrs:
                    ci = tp_hdrs.index(uc)
                    val = str(row[ci]).strip() if len(row) > ci else ""
                    if val: user_vals[mid][uc] = val
    # Wpisz z powrotem do team_pricing_df
    if user_vals and "match_id" in team_pricing_df.columns:
        for uc in USER_COLS:
            if uc not in team_pricing_df.columns:
                team_pricing_df[uc] = ""
        for idx_r, row_r in team_pricing_df.iterrows():
            mid = str(row_r.get("match_id","")).strip()
            if mid in user_vals:
                for uc, val in user_vals[mid].items():
                    team_pricing_df.at[idx_r, uc] = val
    log.info(f"team_pricing: zachowano {sum(len(v) for v in user_vals.values())} ręcznych wpisów")
except Exception as e:
    log.warning(f"Nie udało się zachować ręcznych wpisów w team_pricing: {e}")
results["team_pricing"] = save_df(BOOK, "team_pricing", team_pricing_df)
time.sleep(1.5)
results["players_master"]        = save_df(BOOK, "players_master",        players_master_df)
time.sleep(1.5)
results["expected_lineups"]       = save_df(BOOK, "expected_lineups",       expected_lineups_df)
time.sleep(1.5)
results["team_baselines"]        = save_df(BOOK, "team_baselines",        team_baselines_df)
time.sleep(1.5)
results["live_lineups"]          = save_df(BOOK, "live_lineups",          live_lineups_df)
time.sleep(1.5)
results["alerts_log"]            = save_df(BOOK, "alerts_log",            alerts_df)

ok    = sum(1 for v in results.values() if v)
skipped = sum(1 for v in results.values() if not v)

print(f"\n{'='*50}")
print(f"🎉 Zapis zakończony: {ok} zakładek OK, {skipped} pominiętych")
print(f"{'='*50}")
print(f"\n📊 Podsumowanie DataFrame:")
print(f"   teams_master:          {len(teams_master_df)} drużyn")
print(f"   player_match_features: {len(player_match_features_df)} wierszy")
print(f"   player_rolling:        {len(player_rolling_df)} zawodniczek")
print(f"   team_pricing:          {len(team_pricing_df)} meczów")
print(f"   players_master:        {len(players_master_df)} zawodniczek")
print(f"   expected_lineups:       {len(expected_lineups_df)} slotów")
print(f"   team_baselines:        {len(team_baselines_df)} drużyn")
print(f"   live_lineups:          {len(live_lineups_df)} wierszy (mock)")
print(f"   alerts_log:            {len(alerts_df)} alertów")

In [ ]:
# ═══════════════════════════════════════════════════════════════
# FIX — player_rolling: napraw błędne nazwy drużyn (T_XXXXXXXX)
# Uruchom raz po CELL KOŃCOWYM
# ═══════════════════════════════════════════════════════════════
import time

try: BOOK
except NameError:
    import gspread
    from google.oauth2.service_account import Credentials
    creds = Credentials.from_service_account_file("/content/service_account.json",
        scopes=["https://www.googleapis.com/auth/spreadsheets",
                "https://www.googleapis.com/auth/drive"])
    BOOK = gspread.authorize(creds).open_by_key("1x80zxUeYfKk9_kvsgeR1D9ayoKfaTPE90wNOGvuodnI")

ws   = BOOK.worksheet("player_rolling")
data = ws.get_all_values()
hdrs = data[0]
rows = data[1:]

# Find column indices
def ci(col):
    try: return hdrs.index(col)
    except: return None

i_tid   = ci("team_id")
i_tname = ci("team_name")

if i_tid is None:
    print(f"❌ Brak kolumny team_id — kolumny: {hdrs}")
else:
    if i_tname is None:
        # team_name not yet in sheet — add it based on team_id
        print("ℹ️  Brak kolumny team_name — dodam ją na podstawie team_id")
        # Add header
        ws.update(f"A1", [hdrs + ["team_name"]], value_input_option="USER_ENTERED")
        hdrs = hdrs + ["team_name"]
        i_tname = len(hdrs) - 1
        # Add values for each row
        for j, row in enumerate(rows):
            if len(row) > i_tid:
                canonical = get_canonical_name_by_tid(str(row[i_tid]).strip())
                col_letter = chr(ord("A") + i_tname)
                ws.update(f"{col_letter}{j+2}", [[canonical]],
                          value_input_option="USER_ENTERED")
                time.sleep(0.3)
        print(f"✅ Dodano kolumnę team_name dla {len(rows)} wierszy")
if True:
    # Find rows with hash team names
    updates = []
    for j, row in enumerate(rows):
        if len(row) <= max(i_tid, i_tname): continue
        tid_val   = str(row[i_tid]).strip()
        tname_val = str(row[i_tname]).strip()

        # Fix team_name if it looks like a hash
        import re as _re
        if _re.match(r"^T_[0-9A-F]{8}$", tname_val):
            canonical = get_canonical_name_by_tid(tid_val)
            if canonical and not _re.match(r"^T_[0-9A-F]{8}$", canonical):
                updates.append((j+2, i_tname+1, canonical))  # +2 = header + 1-indexed

    print(f"Znaleziono {len(updates)} wierszy z błędną nazwą drużyny")

    # Apply updates in batches
    if updates:
        for row_n, col_n, val in updates[:5]:  # show sample
            print(f"  Wiersz {row_n}: → '{val}'")
        if len(updates) > 5: print(f"  ... i {len(updates)-5} więcej")

        for row_n, col_n, val in updates:
            col_letter = chr(ord("A") + col_n - 1)
            ws.update(f"{col_letter}{row_n}", [[val]], value_input_option="USER_ENTERED")
            time.sleep(0.3)

        print(f"\n✅ Naprawiono {len(updates)} nazw drużyn w player_rolling")
    else:
        print("✅ Brak błędnych nazw — player_rolling jest czysty")

# Also show still-unknown hashes
import re as _re
unknown_tids = set()
for row in rows:
    if len(row) > i_tname:
        tn = str(row[i_tname]).strip()
        if _re.match(r"^T_[0-9A-F]{8}$", tn):
            unknown_tids.add(tn)

if unknown_tids:
    print(f"\n⚠️  Nadal nieznane team_id: {unknown_tids}")
    print("   Uruchom DEBUG cell żeby zidentyfikować z players_raw")


In [ ]:
# ═══════════════════════════════════════════════════════════════
# CLEANUP — expected_lineups  ← URUCHOM PO CELL KOŃCOWYM
# Czyści i deduplikuje expected_lineups bezpośrednio w Sheets
# ═══════════════════════════════════════════════════════════════
import pandas as pd, unicodedata, hashlib, re, time
from datetime import datetime

try:
    BOOK
except NameError:
    import gspread
    from google.oauth2.service_account import Credentials
    creds = Credentials.from_service_account_file("/content/service_account.json",
        scopes=["https://www.googleapis.com/auth/spreadsheets",
                "https://www.googleapis.com/auth/drive"])
    BOOK = gspread.authorize(creds).open_by_key("1x80zxUeYfKk9_kvsgeR1D9ayoKfaTPE90wNOGvuodnI")

def _n(s):
    s = re.sub(r"\s+", " ", str(s).strip())
    s = unicodedata.normalize("NFD", s)
    return "".join(c for c in s if unicodedata.category(c) != "Mn").title().strip()

def tid(name):
    return "T_" + hashlib.md5(_n(name).encode()).hexdigest()[:8].upper()

C = {
    "Volero Le Cannet W":"Volero Le Cannet W","Volero Le Cannet":"Volero Le Cannet W","Le Cannet W":"Volero Le Cannet W","Le Cannet":"Volero Le Cannet W",
    "Vandoeuvre Nancy W":"Vandoeuvre Nancy W","Vandoeuvre Nancy":"Vandoeuvre Nancy W","Vandoeuvre-Nancy":"Vandoeuvre Nancy W",
    "Volley Mulhouse Alsace W":"Volley Mulhouse Alsace W","Volley Mulhouse Alsace":"Volley Mulhouse Alsace W","Mulhouse W":"Volley Mulhouse Alsace W","Mulhouse":"Volley Mulhouse Alsace W",
    "Terville-Florange W":"Terville-Florange W","Terville Florange W":"Terville-Florange W","Terville-Florange OC":"Terville-Florange W","Terville-Florange":"Terville-Florange W","Terville Florange":"Terville-Florange W",
    "France Avenir W":"France Avenir W","France Avenir":"France Avenir W","France Avenir 2024":"France Avenir W",
    "Beziers W":"Beziers W","Béziers W":"Beziers W","Beziers Volley":"Beziers W","Beziers":"Beziers W","Béziers":"Beziers W",
    "Levallois Paris W":"Levallois Paris W","Levallois Paris":"Levallois Paris W","Levallois Paris SC":"Levallois Paris W",
    "RC Cannes W":"RC Cannes W","RC Cannes":"RC Cannes W","Cannes W":"RC Cannes W","Cannes":"RC Cannes W",
    "Evreux W":"Evreux W","Evreux":"Evreux W",
    "VC Marcq-en-Baroeul W":"VC Marcq-en-Baroeul W","VC Marcq-en-Baroeul":"VC Marcq-en-Baroeul W","Marcq en Baroeul W":"VC Marcq-en-Baroeul W","Marcq en Baroeul":"VC Marcq-en-Baroeul W","Marcq-en-Baroeul W":"VC Marcq-en-Baroeul W","Marcq-en-Baroeul":"VC Marcq-en-Baroeul W",
    "Saint-Die W":"Saint-Die W","Saint-Die":"Saint-Die W","St-Dié W":"Saint-Die W","St-Dié":"Saint-Die W","St-Die W":"Saint-Die W","St-Die":"Saint-Die W","Saint-Die-des-Vosges VB":"Saint-Die W",
    "Bordeaux Merignac W":"Bordeaux Merignac W","Bordeaux Mérignac W":"Bordeaux Merignac W","Bordeaux Merignac":"Bordeaux Merignac W","Bordeaux Mérignac":"Bordeaux Merignac W",
    "VBC Chamalieres W":"VBC Chamalieres W","VBC Chamalieres":"VBC Chamalieres W","Chamalières W":"VBC Chamalieres W","Chamalières":"VBC Chamalieres W","Chamalieres W":"VBC Chamalieres W","Chamalieres":"VBC Chamalieres W",
    "Neptunes de Nantes W":"Neptunes de Nantes W","Neptunes de Nantes":"Neptunes de Nantes W","Nantes W":"Neptunes de Nantes W","Nantes":"Neptunes de Nantes W",
}

def canon(name):
    n = str(name).strip()
    if n in C: return C[n]
    nn = _n(n)
    for k,v in C.items():
        if _n(k)==nn: return v
    return n

# ── Wczytaj surowe dane z Sheets ────────────────────────────────
ws   = BOOK.worksheet("expected_lineups")
data = ws.get_all_values()
hdrs = data[0]
rows = data[1:]
print(f"Wczytano: {len(rows)} wierszy, nagłówki: {hdrs}")

# Indeksy kolumn
def ci(col):
    try: return hdrs.index(col)
    except: return None

i_tname = ci("team_name") or 1
i_teid  = ci("team_id")   or 0
i_pname = ci("player_name") or 4
i_slot  = ci("slot_name") or 5
i_notes = ci("notes") or (len(hdrs)-1)
i_snap  = ci("snapshot_date") or 2

# ── Filtruj złe wiersze ──────────────────────────────────────
BAD_PLAYERS = {"EMPTY","","no_candidate","None","nan","N/A"}
clean = []
removed = 0
for r in rows:
    pname = str(r[i_pname]).strip() if len(r) > i_pname else ""
    notes = str(r[i_notes]).strip() if len(r) > i_notes else ""
    if pname in BAD_PLAYERS or pname == "" or "no_candidate" in notes.lower():
        removed += 1
        continue
    clean.append(r)
print(f"Usunięto {removed} wierszy EMPTY/no_candidate, zostało {len(clean)}")

# ── Napraw nazwy i team_id ────────────────────────────────────
for r in clean:
    old_name = str(r[i_tname]).strip() if len(r) > i_tname else ""
    new_name = canon(old_name)
    if len(r) > i_tname: r[i_tname] = new_name
    if len(r) > i_teid:  r[i_teid]  = tid(new_name)

# ── Deduplikuj: najnowszy snapshot per (team_name, slot_name) ─
seen = {}  # (team_name, slot_name) -> (snap, row)
for r in clean:
    tname = str(r[i_tname]).strip() if len(r) > i_tname else ""
    slot  = str(r[i_slot]).strip()  if len(r) > i_slot  else ""
    snap  = str(r[i_snap]).strip()  if len(r) > i_snap  else ""
    key   = (tname, slot)
    if key not in seen or snap > seen[key][0]:
        seen[key] = (snap, r)

deduped = [v[1] for v in seen.values()]
print(f"Po deduplikacji: {len(deduped)} wierszy")

# ── Sortuj: team_name, potem slot ────────────────────────────
SLOTS = ["S","OPP","OH1","OH2","MB1","MB2","L"]
def slot_k(r):
    s = str(r[i_slot]).strip() if len(r) > i_slot else ""
    return SLOTS.index(s) if s in SLOTS else 99

deduped.sort(key=lambda r: (str(r[i_tname]).strip(), slot_k(r)))

# ── Weryfikuj ─────────────────────────────────────────────────
from collections import defaultdict
team_slots = defaultdict(list)
for r in deduped:
    team_slots[str(r[i_tname]).strip()].append(str(r[i_slot]).strip())

print("\nSloty per drużyna:")
all_ok = True
for t in sorted(team_slots.keys()):
    s = team_slots[t]
    ok = len(s)==7
    if not ok: all_ok = False
    print(f"  {'✅' if ok else '⚠️ '} {t:35s} {len(s)} slotów: {s}")

# ── Zapisz bezpośrednio do Sheets ────────────────────────────
print(f"\n💾 Zapisuję {len(deduped)} wierszy do expected_lineups...")
ws.clear()
time.sleep(2)
ws.update("A1", [hdrs], value_input_option="USER_ENTERED")
time.sleep(1)
for i in range(0, len(deduped), 50):
    ws.append_rows(deduped[i:i+50], value_input_option="USER_ENTERED")
    time.sleep(1)
print(f"✅ Gotowe! {len(deduped)} wierszy, {len(team_slots)} drużyn")
print(f"   Wszystkie po 7 slotów: {'TAK' if all_ok else 'NIE — sprawdź wyżej'}")


---
## 🧪 BONUS — Worked Example: Pełne porównanie składu
Uruchom po wykonaniu wszystkich powyższych komórek.

In [ ]:
def print_worked_example(
    shock_results: pd.DataFrame,
    expected_lineups: pd.DataFrame,
    live_lineups: pd.DataFrame,
) -> None:
    """Printuje czytelny worked example dla pierwszego meczu z shock."""
    if shock_results.empty:
        print('Brak wyników shock do pokazania.')
        return

    # Weź mecz z największym shock
    worst = shock_results.loc[shock_results['delta_TIS_pct'].astype(float).idxmin()]

    match_id = worst['match_id']
    team_id  = worst['team_id']

    print('=' * 70)
    print(f'WORKED EXAMPLE — LINEUP COMPARISON')
    print(f'Mecz:    {match_id}')
    print(f'Drużyna: {worst["team_name"]}')
    print('=' * 70)

    print('\n📋 TYPICAL LINEUP (oczekiwany):')
    typ = expected_lineups[expected_lineups['team_id'] == team_id]
    for _, r in typ.iterrows():
        print(f'  {r["slot_name"]:5s} | {r["player_name"]:30s} | PIS_rolling={r["pis_rolling"]:6.3f} | slot_value={r["slot_value"]:6.3f}')
    print(f'  TIS_typical = {worst["TIS_typical"]:.4f}')

    print('\n🔴 LIVE LINEUP (faktyczny):')
    live = live_lineups[
        (live_lineups['match_id'] == match_id) &
        (live_lineups['team_id'] == team_id)
    ]
    for _, r in live.iterrows():
        missing_flag = '⚠️ BRAK' if r['is_missing_from_typical'] == 'True' else '  OK'
        print(f'  {r["slot_name"]:5s} | {r["live_player_name_normalized"]:30s} | conf={r["resolution_confidence"]:6s} | {missing_flag}')
    print(f'  TIS_live = {worst["TIS_live"]:.4f}')

    print('\n📉 WYNIKI LINEUP SHOCK:')
    print(f'  delta_TIS_abs  = {worst["delta_TIS_abs"]:.4f}')
    print(f'  delta_TIS_pct  = {float(worst["delta_TIS_pct"])*100:.1f}%')
    print(f'  lineup_shock   = {worst["lineup_shock"]}')
    print(f'  shock_level    = {worst["shock_level"].upper()}')
    print(f'  missing_players = {worst["missing_players"] or "(brak)"}')
    print(f'  missing_key_roles = {worst["missing_key_roles"] or "(brak)"}')

    print('\n🎯 KOREKTA ODDS:')
    odds = compute_fair_odds(worst.to_dict(), base_prob=0.50, market_odds=1.95)
    print(f'  base_prob    = {odds["base_prob"]:.2f}')
    print(f'  model_prob   = {odds["model_prob"]:.4f}')
    print(f'  true_odds    = {odds["true_odds"]:.3f}')
    print(f'  market_odds  = {odds["market_odds"]}')
    print(f'  market_prob  = {odds["market_prob"]}')
    print(f'  edge         = {odds["edge"]}')
    print(f'  edge_band    = {odds["edge_band"]}')
    print('=' * 70)

print_worked_example(shock_results_df, expected_lineups_df, live_lineups_df)

---
## 🔮 Placeholders do v1/v2

| Komponent | Status v0 | Plan v1/v2 |
|---|---|---|
| **Live protocol parser** | Mock JSON dict | Scraper z lnv-web.dataproject.com per set_no |
| **Bookmaker odds connector** | `None` / ręczny dict | API Pinnacle / Bet365 / Betfair Exchange |
| **DTR_pre integration** | `base_prob=0.50` placeholder | Serve-receive based historical score |
| **XGBoost probability model** | Deterministyczna korekta delta_TIS | Klasyfikator binarny: P(win)=f(features) |
| **Position assignment** | Tylko libero auto, reszta UNKNOWN | positions_master + ręczny CSV import |
| **Telegram alerts** | Brak | Bot z push-notification per alert |

---
## ✅ Rekomendacja architektury v0

**Wybierz opcję A: Google Sheets + Python (Colab)**

- Wystarczy do PoC i wczesnego MVP
- Zero dodatkowej infrastruktury
- Sheets jako audytowalny "database"
- Przejdź na SQLite/Postgres dopiero gdy alerts_log > ~5000 wierszy lub potrzebujesz sub-sekundowych query